In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!sudo apt-get install -y tabix
!sudo apt-get update -q
!sudo apt-get install bcftools -y -q

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libhtscodecs2
The following NEW packages will be installed:
  libhtscodecs2 tabix
0 upgraded, 2 newly installed, 0 to remove and 2 not upgraded.
Need to get 404 kB of archives.
After this operation, 1,300 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libhtscodecs2 amd64 1.1.1-3 [53.2 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tabix amd64 1.13+ds-2build1 [351 kB]
Fetched 404 kB in 1s (410 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 2.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falli

In [4]:
import subprocess
import os

chr_num = 4
VCF_FILE = f'/content/drive/MyDrive/FYP_DATA/DATA/annotated_data/sg10k/annotated/sg10k_annotated/sg10k_chr{chr_num}_annotated.vcf.gz'
OUTPUT_DIR = '/content/drive/MyDrive/FYP_DATA/DATA/annotated_data/sg10k/annotated/annotated_and_missense_filtered/'
TEMP_VCF   = OUTPUT_DIR + f'sg10k_chr{chr_num}_canonical_missense.vcf'
OUTPUT_VCF = OUTPUT_DIR + f'sg10k_chr{chr_num}_canonical_missense.vcf.gz'

# CSQ column indices — confirmed from inspection
CONSEQUENCE_IDX = 1
CANONICAL_IDX   = 23

kept  = 0
total = 0

print("⏳ Filtering canonical missense variants...")

with open(TEMP_VCF, 'w') as out_f:

    # ── write header unchanged ──────────────────────────────────
    header_result = subprocess.run(
        f"bcftools view -h {VCF_FILE}",
        shell=True, capture_output=True, text=True
    )
    out_f.write(header_result.stdout)

    # ── stream variants ─────────────────────────────────────────
    proc = subprocess.Popen(
        f"bcftools view -H {VCF_FILE}",
        shell=True, stdout=subprocess.PIPE, text=True
    )

    for line in proc.stdout:
        total += 1
        fields = line.strip().split('\t')
        info   = fields[7]

        if 'CSQ=' not in info:
            continue

        csq_blob    = info.split('CSQ=')[1].split(';')[0]
        transcripts = csq_blob.split(',')

        for transcript in transcripts:
            cols = transcript.split('|')
            if len(cols) <= CANONICAL_IDX:
                continue
            consequence = cols[CONSEQUENCE_IDX]
            canonical   = cols[CANONICAL_IDX]
            if canonical == 'YES' and 'missense_variant' in consequence:
                out_f.write(line)
                kept += 1
                break

        if total % 100_000 == 0:
            print(f"   Processed {total:,} variants, kept {kept:,} so far...")

    proc.wait()

print(f"\n✅ Done! Processed {total:,} total variants")
print(f"   Kept {kept:,} canonical missense variants")

# ── Compress ────────────────────────────────────────────────────
print(f"\n⏳ Compressing...")
bgzip_result = subprocess.run(
    f"bgzip -f {TEMP_VCF}",
    shell=True, capture_output=True, text=True
)
print(f"bgzip return code : {bgzip_result.returncode}")
print(f"bgzip stderr      : {bgzip_result.stderr.strip() or 'none'}")

# ── Index ────────────────────────────────────────────────────────
print(f"\n⏳ Indexing...")
subprocess.run(f"bcftools index -t -f {OUTPUT_VCF}", shell=True, capture_output=True)
print(f"✅ Indexed!")

# ── Verify ───────────────────────────────────────────────────────
print(f"\nCompressed file exists : {os.path.exists(OUTPUT_VCF)}")
print(f"Index file exists      : {os.path.exists(OUTPUT_VCF + '.tbi')}")

count_result = subprocess.run(
    f"bcftools index -n {OUTPUT_VCF}",
    shell=True, capture_output=True, text=True
)
n_variants = int(count_result.stdout.strip())
input_mb   = os.path.getsize(VCF_FILE)  / (1024**2)
output_mb  = os.path.getsize(OUTPUT_VCF) / (1024**2)

print("=" * 60)
print("VERIFICATION")
print("=" * 60)
print(f"Input size             : {input_mb:,.2f} MB")
print(f"Output size            : {output_mb:,.2f} MB")
print(f"Canonical missense kept: {n_variants:,}")
print("=" * 60)

# --------------------------- QUALITY CHECKS ------------------------------ #
import subprocess
import os
from collections import defaultdict

# ── Configuration ────────────────────────────────────────────────────
OUTPUT_VCF = f'/content/drive/MyDrive/FYP_DATA/DATA/annotated_data/sg10k/annotated/annotated_and_missense_filtered/sg10k_chr{chr_num}_canonical_missense.vcf.gz'
VCF_FILE   = f'/content/drive/MyDrive/FYP_DATA/DATA/annotated_data/sg10k/annotated/sg10k_annotated/sg10k_chr{chr_num}_annotated.vcf.gz'

CONSEQUENCE_IDX    = 1
CANONICAL_IDX      = 23
IMPACT_IDX         = 2
SYMBOL_IDX         = 3
PROTEIN_POS_IDX    = 14
AMINO_ACIDS_IDX    = 15

passed = []
failed = []
warnings = []

def check(name, condition, detail=""):
    if condition:
        passed.append(name)
        print(f"   ✅ {name}")
    else:
        failed.append(name)
        print(f"   ❌ {name}")
    if detail:
        print(f"      → {detail}")

def warn(name, detail=""):
    warnings.append(name)
    print(f"   ⚠️  {name}")
    if detail:
        print(f"      → {detail}")

print("=" * 80)
print("QUALITY CHECK — sg10k_chr22_canonical_missense.vcf.gz")
print("=" * 80)

# ════════════════════════════════════════════════════════════════════
# CHECK 1: FILE & INDEX
# ════════════════════════════════════════════════════════════════════
print("\n📁 CHECK 1: FILE & INDEX")
print("-" * 60)

file_ok  = os.path.exists(OUTPUT_VCF)
index_ok = os.path.exists(OUTPUT_VCF + '.tbi')
size_mb  = os.path.getsize(OUTPUT_VCF) / (1024**2) if file_ok else 0

check("File exists",       file_ok,  f"Size: {size_mb:.2f} MB")
check("Index exists",      index_ok)
check("File is not empty", size_mb > 0.1, f"{size_mb:.2f} MB")

# ════════════════════════════════════════════════════════════════════
# CHECK 2: VARIANT COUNT
# ════════════════════════════════════════════════════════════════════
print("\n🔢 CHECK 2: VARIANT COUNT")
print("-" * 60)

count_result = subprocess.run(
    f"bcftools index -n {OUTPUT_VCF}",
    shell=True, capture_output=True, text=True
)
n_variants = int(count_result.stdout.strip()) if count_result.stdout.strip() else 0
print(f"   Total canonical missense variants : {n_variants:,}")

check("Variant count matches filtering output", n_variants == 12_998,
      f"Got {n_variants:,}, expected 12,998")
check("Count is lower than any-transcript missense (14,404)", n_variants < 14_404,
      f"{n_variants:,} < 14,404 ✓ — canonical filter correctly reduced count")
check("Count in plausible range (8k–16k)", 8_000 <= n_variants <= 16_000,
      f"{n_variants:,} variants")

# ════════════════════════════════════════════════════════════════════
# CHECK 3: SAMPLE COUNT
# SG10K-specific — genotypes must be preserved for all 1,125 samples
# ════════════════════════════════════════════════════════════════════
print("\n👥 CHECK 3: SAMPLE COUNT (SG10K-specific)")
print("-" * 60)

samples_result = subprocess.run(
    f"bcftools query -l {OUTPUT_VCF} | wc -l",
    shell=True, capture_output=True, text=True
)
n_samples = int(samples_result.stdout.strip()) if samples_result.stdout.strip() else 0
print(f"   Samples in filtered file : {n_samples:,}")

check("Sample count = 1,125", n_samples == 1_125,
      f"Got {n_samples} — filtering must not remove samples")

# ════════════════════════════════════════════════════════════════════
# CHECK 4: CHROMOSOME INTEGRITY
# ════════════════════════════════════════════════════════════════════
print("\n🧬 CHECK 4: CHROMOSOME INTEGRITY")
print("-" * 60)

chr_result = subprocess.run(
    f"bcftools query -f '%CHROM\n' {OUTPUT_VCF} | sort -u",
    shell=True, capture_output=True, text=True
)
chromosomes = [c for c in chr_result.stdout.strip().split('\n') if c]
print(f"   Chromosomes found: {chromosomes}")

check("Only chr22 present",          chromosomes == ['chr22'], f"Found: {chromosomes}")
check("Chromosome has 'chr' prefix", all(c.startswith('chr') for c in chromosomes))

# ════════════════════════════════════════════════════════════════════
# CHECK 5: CANONICAL MISSENSE INTEGRITY
# Scan every variant — both conditions must be on the SAME transcript
# ════════════════════════════════════════════════════════════════════
print("\n🎯 CHECK 5: CANONICAL MISSENSE INTEGRITY (scanning all variants)")
print("-" * 60)
print("   Scanning every variant...")

non_missense_count  = 0
non_canonical_count = 0
cross_transcript_fp = 0
valid_count         = 0
no_csq_count        = 0

proc = subprocess.Popen(
    f"bcftools view -H {OUTPUT_VCF}",
    shell=True, stdout=subprocess.PIPE, text=True
)
for line in proc.stdout:
    fields = line.strip().split('\t')
    info   = fields[7]

    if 'CSQ=' not in info:
        no_csq_count += 1
        continue

    csq_blob    = info.split('CSQ=')[1].split(';')[0]
    transcripts = csq_blob.split(',')

    found_canonical       = False
    canonical_is_missense = False

    for t in transcripts:
        cols = t.split('|')
        if len(cols) <= CANONICAL_IDX:
            continue
        if cols[CANONICAL_IDX] == 'YES':
            found_canonical = True
            if 'missense_variant' in cols[CONSEQUENCE_IDX]:
                canonical_is_missense = True
                break

    if not found_canonical:
        non_canonical_count += 1
    elif not canonical_is_missense:
        non_missense_count += 1
        for t in transcripts:
            cols = t.split('|')
            if len(cols) > CANONICAL_IDX and cols[CANONICAL_IDX] != 'YES':
                if 'missense_variant' in cols[CONSEQUENCE_IDX]:
                    cross_transcript_fp += 1
                    break
    else:
        valid_count += 1
proc.wait()

print(f"   Fully valid (canonical=YES + missense): {valid_count:,}")
print(f"   No canonical transcript found          : {non_canonical_count:,}")
print(f"   Canonical found but NOT missense       : {non_missense_count:,}")
print(f"   Cross-transcript false positives       : {cross_transcript_fp:,}")
print(f"   No CSQ field                           : {no_csq_count:,}")

check("All variants have CANONICAL=YES transcript",  non_canonical_count == 0)
check("All variants are missense in canonical",       non_missense_count == 0)
check("Zero cross-transcript false positives",        cross_transcript_fp == 0)
check("No variants missing CSQ field",                no_csq_count == 0)
check("Valid count matches total",                    valid_count == n_variants,
      f"{valid_count:,} valid out of {n_variants:,}")

# ════════════════════════════════════════════════════════════════════
# CHECK 6: IMPACT DISTRIBUTION
# All canonical missense should be MODERATE
# ════════════════════════════════════════════════════════════════════
print("\n⚡ CHECK 6: IMPACT DISTRIBUTION")
print("-" * 60)

impact_counts = defaultdict(int)
proc = subprocess.Popen(f"bcftools view -H {OUTPUT_VCF}", shell=True, stdout=subprocess.PIPE, text=True)
for line in proc.stdout:
    info = line.strip().split('\t')[7]
    if 'CSQ=' not in info:
        continue
    for t in info.split('CSQ=')[1].split(';')[0].split(','):
        cols = t.split('|')
        if len(cols) > CANONICAL_IDX and cols[CANONICAL_IDX] == 'YES':
            if 'missense_variant' in cols[CONSEQUENCE_IDX]:
                impact_counts[cols[IMPACT_IDX]] += 1
                break
proc.wait()

print("   Impact breakdown:")
for impact, count in sorted(impact_counts.items(), key=lambda x: -x[1]):
    print(f"      {impact:12s}: {count:,}")

check("All canonical missense are MODERATE impact",
      set(impact_counts.keys()) == {'MODERATE'},
      f"Found: {set(impact_counts.keys())}")

# ════════════════════════════════════════════════════════════════════
# CHECK 7: INFO FIELDS PRESERVED (SG10K-specific)
# AF, AC, AN, AR2, DR2 must all be present
# ════════════════════════════════════════════════════════════════════
print("\n📊 CHECK 7: SG10K INFO FIELDS PRESERVED")
print("-" * 60)

peek = subprocess.run(
    f"bcftools view -H {OUTPUT_VCF} | head -1",
    shell=True, capture_output=True, text=True
).stdout.strip()

if peek:
    info_field = peek.split('\t')[7] if len(peek.split('\t')) > 7 else ''
    for field in ['AF=', 'AC=', 'AN=', 'AR2=', 'DR2=', 'CSQ=']:
        present = field in info_field
        check(f"{field.rstrip('=')} field present", present)
else:
    print("   ❌ Could not read first variant")

# ════════════════════════════════════════════════════════════════════
# CHECK 8: IMPUTATION QUALITY (SG10K-specific)
# AR2 and DR2 are imputation quality scores — check their distribution
# ════════════════════════════════════════════════════════════════════
print("\n🔬 CHECK 8: IMPUTATION QUALITY (AR2 / DR2)")
print("-" * 60)

ar2_result = subprocess.run(
    f"bcftools query -f '%INFO/AR2\n' {OUTPUT_VCF}",
    shell=True, capture_output=True, text=True
)
dr2_result = subprocess.run(
    f"bcftools query -f '%INFO/DR2\n' {OUTPUT_VCF}",
    shell=True, capture_output=True, text=True
)

ar2_vals = [float(v) for v in ar2_result.stdout.strip().split('\n') if v and v != '.']
dr2_vals = [float(v) for v in dr2_result.stdout.strip().split('\n') if v and v != '.']

if ar2_vals:
    ar2_mean = sum(ar2_vals) / len(ar2_vals)
    ar2_low  = sum(1 for v in ar2_vals if v < 0.3)
    print(f"   AR2 — mean: {ar2_mean:.3f}  low quality (<0.3): {ar2_low:,} ({ar2_low/len(ar2_vals)*100:.1f}%)")
    check("AR2 mean > 0.7 (good imputation quality)", ar2_mean > 0.7, f"Mean AR2 = {ar2_mean:.3f}")
    if ar2_low > 0:
        warn(f"{ar2_low:,} variants have AR2 < 0.3 (low imputation quality)",
             "Consider filtering these out in a later step if needed")
else:
    print("   AR2 field not populated")

if dr2_vals:
    dr2_mean = sum(dr2_vals) / len(dr2_vals)
    dr2_low  = sum(1 for v in dr2_vals if v < 0.3)
    print(f"   DR2 — mean: {dr2_mean:.3f}  low quality (<0.3): {dr2_low:,} ({dr2_low/len(dr2_vals)*100:.1f}%)")
    check("DR2 mean > 0.7 (good imputation quality)", dr2_mean > 0.7, f"Mean DR2 = {dr2_mean:.3f}")
else:
    print("   DR2 field not populated")

# ════════════════════════════════════════════════════════════════════
# CHECK 9: ALLELE FREQUENCY DISTRIBUTION (SG10K-specific)
# AF recalculated for Indian subset — check it makes sense
# ════════════════════════════════════════════════════════════════════
print("\n📈 CHECK 9: ALLELE FREQUENCY DISTRIBUTION")
print("-" * 60)

af_result = subprocess.run(
    f"bcftools query -f '%INFO/AF\n' {OUTPUT_VCF}",
    shell=True, capture_output=True, text=True
)
af_vals = [float(v) for v in af_result.stdout.strip().split('\n') if v and v != '.']

if af_vals:
    af_zero     = sum(1 for v in af_vals if v == 0)
    af_rare     = sum(1 for v in af_vals if 0 < v < 0.01)
    af_common   = sum(1 for v in af_vals if v >= 0.01)
    af_max      = max(af_vals)
    af_mean     = sum(af_vals) / len(af_vals)

    print(f"   AF = 0 (not observed in Indian subset) : {af_zero:,}  ({af_zero/len(af_vals)*100:.1f}%)")
    print(f"   AF > 0 but < 0.01 (rare)               : {af_rare:,}  ({af_rare/len(af_vals)*100:.1f}%)")
    print(f"   AF >= 0.01 (common)                     : {af_common:,} ({af_common/len(af_vals)*100:.1f}%)")
    print(f"   Max AF                                  : {af_max:.4f}")
    print(f"   Mean AF                                 : {af_mean:.6f}")

    check("AF values are between 0 and 1", all(0 <= v <= 1 for v in af_vals))
    check("No AF values above 1.0 (invalid)", all(v <= 1.0 for v in af_vals))
    if af_zero > n_variants * 0.5:
        warn(f"{af_zero:,} variants have AF=0 in Indian subset",
             "These variants exist in gnomAD but were not observed in SG10K Indian samples")

# ════════════════════════════════════════════════════════════════════
# CHECK 10: GENOTYPE FORMAT (SG10K-specific)
# SG10K has actual genotypes — verify FORMAT field and GT is present
# ════════════════════════════════════════════════════════════════════
print("\n🧪 CHECK 10: GENOTYPE FORMAT (SG10K-specific)")
print("-" * 60)

# Check FORMAT field in header
fmt_result = subprocess.run(
    f"bcftools view -h {OUTPUT_VCF} | grep '##FORMAT=<ID=GT'",
    shell=True, capture_output=True, text=True
)
gt_in_header = bool(fmt_result.stdout.strip())
check("GT FORMAT field in header", gt_in_header)

# Check actual genotype columns present
cols_result = subprocess.run(
    f"bcftools view -H {OUTPUT_VCF} | head -1 | awk '{{print NF}}'",
    shell=True, capture_output=True, text=True
)
n_cols = int(cols_result.stdout.strip()) if cols_result.stdout.strip() else 0
expected_cols = 9 + 1125  # 9 fixed VCF columns + 1125 sample columns
print(f"   Columns per row : {n_cols:,} (expected {expected_cols:,})")
check(f"Column count = 9 + 1,125 samples = {expected_cols}", n_cols == expected_cols,
      f"Got {n_cols}, expected {expected_cols}")

# Check a sample genotype looks valid
gt_result = subprocess.run(
    f"bcftools query -f '[%GT ]\\n' {OUTPUT_VCF} | head -1 | tr ' ' '\\n' | sort | uniq -c | sort -rn | head -5",
    shell=True, capture_output=True, text=True
)
print(f"   Most common genotypes (first variant):")
for line in gt_result.stdout.strip().split('\n')[:5]:
    print(f"      {line.strip()}")

# ════════════════════════════════════════════════════════════════════
# CHECK 11: GENE COVERAGE
# ════════════════════════════════════════════════════════════════════
print("\n🧫 CHECK 11: GENE COVERAGE")
print("-" * 60)

gene_counts = defaultdict(int)
protein_pos_missing = 0
amino_acids_missing = 0

proc = subprocess.Popen(f"bcftools view -H {OUTPUT_VCF}", shell=True, stdout=subprocess.PIPE, text=True)
for line in proc.stdout:
    info = line.strip().split('\t')[7]
    if 'CSQ=' not in info:
        continue
    for t in info.split('CSQ=')[1].split(';')[0].split(','):
        cols = t.split('|')
        if len(cols) > CANONICAL_IDX and cols[CANONICAL_IDX] == 'YES':
            if 'missense_variant' in cols[CONSEQUENCE_IDX]:
                gene = cols[SYMBOL_IDX] if len(cols) > SYMBOL_IDX else ''
                gene_counts[gene] += 1
                if not (cols[PROTEIN_POS_IDX] if len(cols) > PROTEIN_POS_IDX else ''):
                    protein_pos_missing += 1
                if not (cols[AMINO_ACIDS_IDX] if len(cols) > AMINO_ACIDS_IDX else ''):
                    amino_acids_missing += 1
                break
proc.wait()

n_genes      = len(gene_counts)
avg_per_gene = n_variants / n_genes if n_genes > 0 else 0
empty_genes  = sum(1 for g in gene_counts if g in ('', 'UNKNOWN'))
top_genes    = sorted(gene_counts.items(), key=lambda x: -x[1])[:5]

print(f"   Unique genes covered        : {n_genes:,}")
print(f"   Avg variants per gene       : {avg_per_gene:.1f}")
print(f"   Empty gene symbols          : {empty_genes:,}")
print(f"   Missing Protein_position    : {protein_pos_missing:,}")
print(f"   Missing Amino_acids         : {amino_acids_missing:,}")
print(f"   Top 5 genes by variant count:")
for gene, count in top_genes:
    print(f"      {gene or '(empty)':20s}: {count:,}")

check("Gene count in plausible range (300–700)", 300 <= n_genes <= 700, f"{n_genes} genes")
check("No excessive empty gene symbols", empty_genes < n_variants * 0.05)
check("Protein_position present for all variants", protein_pos_missing == 0,
      f"{protein_pos_missing:,} missing — needed for sequence generation")
check("Amino_acids present for all variants", amino_acids_missing == 0,
      f"{amino_acids_missing:,} missing — needed for sequence generation")

# ════════════════════════════════════════════════════════════════════
# CHECK 12: DUPLICATE POSITIONS
# ════════════════════════════════════════════════════════════════════
print("\n🔁 CHECK 12: DUPLICATE POSITIONS")
print("-" * 60)

dup_result = subprocess.run(
    f"bcftools query -f '%CHROM:%POS:%REF:%ALT\n' {OUTPUT_VCF} | sort | uniq -d | wc -l",
    shell=True, capture_output=True, text=True
)
n_dups = int(dup_result.stdout.strip()) if dup_result.stdout.strip() else 0
print(f"   Duplicate positions: {n_dups:,}")
check("No duplicate variant positions", n_dups == 0)

# ════════════════════════════════════════════════════════════════════
# CHECK 13: VCF FORMAT INTEGRITY
# ════════════════════════════════════════════════════════════════════
print("\n📋 CHECK 13: VCF FORMAT INTEGRITY")
print("-" * 60)

header_lines = subprocess.run(
    f"bcftools view -h {OUTPUT_VCF} | wc -l",
    shell=True, capture_output=True, text=True
).stdout.strip()
n_header = int(header_lines) if header_lines else 0
print(f"   Header lines: {n_header}")
check("Header present and substantial", n_header > 10)

malformed = subprocess.run(
    f"bcftools view -H {OUTPUT_VCF} | awk 'NF<9' | wc -l",
    shell=True, capture_output=True, text=True
).stdout.strip()
n_malformed = int(malformed) if malformed else 0
check("No malformed variant lines (< 9 columns)", n_malformed == 0,
      f"SG10K needs 9 + 1125 sample columns per line")

empty_alleles = subprocess.run(
    f"bcftools query -f '%REF\t%ALT\n' {OUTPUT_VCF} | awk '$1==\".\" || $2==\".\"' | wc -l",
    shell=True, capture_output=True, text=True
).stdout.strip()
n_empty = int(empty_alleles) if empty_alleles else 0
check("All variants have REF and ALT alleles", n_empty == 0)

# ════════════════════════════════════════════════════════════════════
# FINAL REPORT
# ════════════════════════════════════════════════════════════════════
print("\n" + "=" * 80)
print("QUALITY CHECK REPORT — sg10k_chr22_canonical_missense.vcf.gz")
print("=" * 80)
print(f"\n   ✅ Passed   : {len(passed)}")
print(f"   ❌ Failed   : {len(failed)}")
print(f"   ⚠️  Warnings : {len(warnings)}")

if failed:
    print("\n   Failed checks:")
    for f in failed:
        print(f"      ❌ {f}")
if warnings:
    print("\n   Warnings (non-blocking):")
    for w in warnings:
        print(f"      ⚠️  {w}")

if len(failed) == 0:
    print("""
🎉 ALL CHECKS PASSED!
   sg10k_chr22_canonical_missense.vcf.gz is clean and ready.
   Safe to proceed with remaining chromosomes.
""")
elif len(failed) <= 2:
    print("\n⚠️  MOSTLY PASSED — review failed checks above before proceeding.")
else:
    print("\n❌ MULTIPLE CHECKS FAILED — investigate before proceeding.")

print("=" * 80)

⏳ Filtering canonical missense variants...
   Processed 100,000 variants, kept 1,485 so far...
   Processed 200,000 variants, kept 2,447 so far...
   Processed 300,000 variants, kept 3,128 so far...
   Processed 400,000 variants, kept 3,804 so far...
   Processed 500,000 variants, kept 4,107 so far...
   Processed 600,000 variants, kept 4,316 so far...
   Processed 700,000 variants, kept 4,660 so far...
   Processed 800,000 variants, kept 4,932 so far...
   Processed 900,000 variants, kept 5,044 so far...
   Processed 1,000,000 variants, kept 5,238 so far...
   Processed 1,100,000 variants, kept 5,566 so far...
   Processed 1,200,000 variants, kept 5,566 so far...
   Processed 1,300,000 variants, kept 5,652 so far...
   Processed 1,400,000 variants, kept 5,652 so far...
   Processed 1,500,000 variants, kept 6,098 so far...
   Processed 1,600,000 variants, kept 6,732 so far...
   Processed 1,700,000 variants, kept 6,917 so far...
   Processed 1,800,000 variants, kept 7,071 so far...
   

In [5]:
import subprocess
import os

chr_num = 7
VCF_FILE = f'/content/drive/MyDrive/FYP_DATA/DATA/annotated_data/sg10k/annotated/sg10k_annotated/sg10k_chr{chr_num}_annotated.vcf.gz'
OUTPUT_DIR = '/content/drive/MyDrive/FYP_DATA/DATA/annotated_data/sg10k/annotated/annotated_and_missense_filtered/'
TEMP_VCF   = OUTPUT_DIR + f'sg10k_chr{chr_num}_canonical_missense.vcf'
OUTPUT_VCF = OUTPUT_DIR + f'sg10k_chr{chr_num}_canonical_missense.vcf.gz'

# CSQ column indices — confirmed from inspection
CONSEQUENCE_IDX = 1
CANONICAL_IDX   = 23

kept  = 0
total = 0

print("⏳ Filtering canonical missense variants...")

with open(TEMP_VCF, 'w') as out_f:

    # ── write header unchanged ──────────────────────────────────
    header_result = subprocess.run(
        f"bcftools view -h {VCF_FILE}",
        shell=True, capture_output=True, text=True
    )
    out_f.write(header_result.stdout)

    # ── stream variants ─────────────────────────────────────────
    proc = subprocess.Popen(
        f"bcftools view -H {VCF_FILE}",
        shell=True, stdout=subprocess.PIPE, text=True
    )

    for line in proc.stdout:
        total += 1
        fields = line.strip().split('\t')
        info   = fields[7]

        if 'CSQ=' not in info:
            continue

        csq_blob    = info.split('CSQ=')[1].split(';')[0]
        transcripts = csq_blob.split(',')

        for transcript in transcripts:
            cols = transcript.split('|')
            if len(cols) <= CANONICAL_IDX:
                continue
            consequence = cols[CONSEQUENCE_IDX]
            canonical   = cols[CANONICAL_IDX]
            if canonical == 'YES' and 'missense_variant' in consequence:
                out_f.write(line)
                kept += 1
                break

        if total % 100_000 == 0:
            print(f"   Processed {total:,} variants, kept {kept:,} so far...")

    proc.wait()

print(f"\n✅ Done! Processed {total:,} total variants")
print(f"   Kept {kept:,} canonical missense variants")

# ── Compress ────────────────────────────────────────────────────
print(f"\n⏳ Compressing...")
bgzip_result = subprocess.run(
    f"bgzip -f {TEMP_VCF}",
    shell=True, capture_output=True, text=True
)
print(f"bgzip return code : {bgzip_result.returncode}")
print(f"bgzip stderr      : {bgzip_result.stderr.strip() or 'none'}")

# ── Index ────────────────────────────────────────────────────────
print(f"\n⏳ Indexing...")
subprocess.run(f"bcftools index -t -f {OUTPUT_VCF}", shell=True, capture_output=True)
print(f"✅ Indexed!")

# ── Verify ───────────────────────────────────────────────────────
print(f"\nCompressed file exists : {os.path.exists(OUTPUT_VCF)}")
print(f"Index file exists      : {os.path.exists(OUTPUT_VCF + '.tbi')}")

count_result = subprocess.run(
    f"bcftools index -n {OUTPUT_VCF}",
    shell=True, capture_output=True, text=True
)
n_variants = int(count_result.stdout.strip())
input_mb   = os.path.getsize(VCF_FILE)  / (1024**2)
output_mb  = os.path.getsize(OUTPUT_VCF) / (1024**2)

print("=" * 60)
print("VERIFICATION")
print("=" * 60)
print(f"Input size             : {input_mb:,.2f} MB")
print(f"Output size            : {output_mb:,.2f} MB")
print(f"Canonical missense kept: {n_variants:,}")
print("=" * 60)

# --------------------------- QUALITY CHECKS ------------------------------ #
import subprocess
import os
from collections import defaultdict

# ── Configuration ────────────────────────────────────────────────────
OUTPUT_VCF = f'/content/drive/MyDrive/FYP_DATA/DATA/annotated_data/sg10k/annotated/annotated_and_missense_filtered/sg10k_chr{chr_num}_canonical_missense.vcf.gz'
VCF_FILE   = f'/content/drive/MyDrive/FYP_DATA/DATA/annotated_data/sg10k/annotated/sg10k_annotated/sg10k_chr{chr_num}_annotated.vcf.gz'

CONSEQUENCE_IDX    = 1
CANONICAL_IDX      = 23
IMPACT_IDX         = 2
SYMBOL_IDX         = 3
PROTEIN_POS_IDX    = 14
AMINO_ACIDS_IDX    = 15

passed = []
failed = []
warnings = []

def check(name, condition, detail=""):
    if condition:
        passed.append(name)
        print(f"   ✅ {name}")
    else:
        failed.append(name)
        print(f"   ❌ {name}")
    if detail:
        print(f"      → {detail}")

def warn(name, detail=""):
    warnings.append(name)
    print(f"   ⚠️  {name}")
    if detail:
        print(f"      → {detail}")

print("=" * 80)
print("QUALITY CHECK — sg10k_chr22_canonical_missense.vcf.gz")
print("=" * 80)

# ════════════════════════════════════════════════════════════════════
# CHECK 1: FILE & INDEX
# ════════════════════════════════════════════════════════════════════
print("\n📁 CHECK 1: FILE & INDEX")
print("-" * 60)

file_ok  = os.path.exists(OUTPUT_VCF)
index_ok = os.path.exists(OUTPUT_VCF + '.tbi')
size_mb  = os.path.getsize(OUTPUT_VCF) / (1024**2) if file_ok else 0

check("File exists",       file_ok,  f"Size: {size_mb:.2f} MB")
check("Index exists",      index_ok)
check("File is not empty", size_mb > 0.1, f"{size_mb:.2f} MB")

# ════════════════════════════════════════════════════════════════════
# CHECK 2: VARIANT COUNT
# ════════════════════════════════════════════════════════════════════
print("\n🔢 CHECK 2: VARIANT COUNT")
print("-" * 60)

count_result = subprocess.run(
    f"bcftools index -n {OUTPUT_VCF}",
    shell=True, capture_output=True, text=True
)
n_variants = int(count_result.stdout.strip()) if count_result.stdout.strip() else 0
print(f"   Total canonical missense variants : {n_variants:,}")

check("Variant count matches filtering output", n_variants == 12_998,
      f"Got {n_variants:,}, expected 12,998")
check("Count is lower than any-transcript missense (14,404)", n_variants < 14_404,
      f"{n_variants:,} < 14,404 ✓ — canonical filter correctly reduced count")
check("Count in plausible range (8k–16k)", 8_000 <= n_variants <= 16_000,
      f"{n_variants:,} variants")

# ════════════════════════════════════════════════════════════════════
# CHECK 3: SAMPLE COUNT
# SG10K-specific — genotypes must be preserved for all 1,125 samples
# ════════════════════════════════════════════════════════════════════
print("\n👥 CHECK 3: SAMPLE COUNT (SG10K-specific)")
print("-" * 60)

samples_result = subprocess.run(
    f"bcftools query -l {OUTPUT_VCF} | wc -l",
    shell=True, capture_output=True, text=True
)
n_samples = int(samples_result.stdout.strip()) if samples_result.stdout.strip() else 0
print(f"   Samples in filtered file : {n_samples:,}")

check("Sample count = 1,125", n_samples == 1_125,
      f"Got {n_samples} — filtering must not remove samples")

# ════════════════════════════════════════════════════════════════════
# CHECK 4: CHROMOSOME INTEGRITY
# ════════════════════════════════════════════════════════════════════
print("\n🧬 CHECK 4: CHROMOSOME INTEGRITY")
print("-" * 60)

chr_result = subprocess.run(
    f"bcftools query -f '%CHROM\n' {OUTPUT_VCF} | sort -u",
    shell=True, capture_output=True, text=True
)
chromosomes = [c for c in chr_result.stdout.strip().split('\n') if c]
print(f"   Chromosomes found: {chromosomes}")

check("Only chr22 present",          chromosomes == ['chr22'], f"Found: {chromosomes}")
check("Chromosome has 'chr' prefix", all(c.startswith('chr') for c in chromosomes))

# ════════════════════════════════════════════════════════════════════
# CHECK 5: CANONICAL MISSENSE INTEGRITY
# Scan every variant — both conditions must be on the SAME transcript
# ════════════════════════════════════════════════════════════════════
print("\n🎯 CHECK 5: CANONICAL MISSENSE INTEGRITY (scanning all variants)")
print("-" * 60)
print("   Scanning every variant...")

non_missense_count  = 0
non_canonical_count = 0
cross_transcript_fp = 0
valid_count         = 0
no_csq_count        = 0

proc = subprocess.Popen(
    f"bcftools view -H {OUTPUT_VCF}",
    shell=True, stdout=subprocess.PIPE, text=True
)
for line in proc.stdout:
    fields = line.strip().split('\t')
    info   = fields[7]

    if 'CSQ=' not in info:
        no_csq_count += 1
        continue

    csq_blob    = info.split('CSQ=')[1].split(';')[0]
    transcripts = csq_blob.split(',')

    found_canonical       = False
    canonical_is_missense = False

    for t in transcripts:
        cols = t.split('|')
        if len(cols) <= CANONICAL_IDX:
            continue
        if cols[CANONICAL_IDX] == 'YES':
            found_canonical = True
            if 'missense_variant' in cols[CONSEQUENCE_IDX]:
                canonical_is_missense = True
                break

    if not found_canonical:
        non_canonical_count += 1
    elif not canonical_is_missense:
        non_missense_count += 1
        for t in transcripts:
            cols = t.split('|')
            if len(cols) > CANONICAL_IDX and cols[CANONICAL_IDX] != 'YES':
                if 'missense_variant' in cols[CONSEQUENCE_IDX]:
                    cross_transcript_fp += 1
                    break
    else:
        valid_count += 1
proc.wait()

print(f"   Fully valid (canonical=YES + missense): {valid_count:,}")
print(f"   No canonical transcript found          : {non_canonical_count:,}")
print(f"   Canonical found but NOT missense       : {non_missense_count:,}")
print(f"   Cross-transcript false positives       : {cross_transcript_fp:,}")
print(f"   No CSQ field                           : {no_csq_count:,}")

check("All variants have CANONICAL=YES transcript",  non_canonical_count == 0)
check("All variants are missense in canonical",       non_missense_count == 0)
check("Zero cross-transcript false positives",        cross_transcript_fp == 0)
check("No variants missing CSQ field",                no_csq_count == 0)
check("Valid count matches total",                    valid_count == n_variants,
      f"{valid_count:,} valid out of {n_variants:,}")

# ════════════════════════════════════════════════════════════════════
# CHECK 6: IMPACT DISTRIBUTION
# All canonical missense should be MODERATE
# ════════════════════════════════════════════════════════════════════
print("\n⚡ CHECK 6: IMPACT DISTRIBUTION")
print("-" * 60)

impact_counts = defaultdict(int)
proc = subprocess.Popen(f"bcftools view -H {OUTPUT_VCF}", shell=True, stdout=subprocess.PIPE, text=True)
for line in proc.stdout:
    info = line.strip().split('\t')[7]
    if 'CSQ=' not in info:
        continue
    for t in info.split('CSQ=')[1].split(';')[0].split(','):
        cols = t.split('|')
        if len(cols) > CANONICAL_IDX and cols[CANONICAL_IDX] == 'YES':
            if 'missense_variant' in cols[CONSEQUENCE_IDX]:
                impact_counts[cols[IMPACT_IDX]] += 1
                break
proc.wait()

print("   Impact breakdown:")
for impact, count in sorted(impact_counts.items(), key=lambda x: -x[1]):
    print(f"      {impact:12s}: {count:,}")

check("All canonical missense are MODERATE impact",
      set(impact_counts.keys()) == {'MODERATE'},
      f"Found: {set(impact_counts.keys())}")

# ════════════════════════════════════════════════════════════════════
# CHECK 7: INFO FIELDS PRESERVED (SG10K-specific)
# AF, AC, AN, AR2, DR2 must all be present
# ════════════════════════════════════════════════════════════════════
print("\n📊 CHECK 7: SG10K INFO FIELDS PRESERVED")
print("-" * 60)

peek = subprocess.run(
    f"bcftools view -H {OUTPUT_VCF} | head -1",
    shell=True, capture_output=True, text=True
).stdout.strip()

if peek:
    info_field = peek.split('\t')[7] if len(peek.split('\t')) > 7 else ''
    for field in ['AF=', 'AC=', 'AN=', 'AR2=', 'DR2=', 'CSQ=']:
        present = field in info_field
        check(f"{field.rstrip('=')} field present", present)
else:
    print("   ❌ Could not read first variant")

# ════════════════════════════════════════════════════════════════════
# CHECK 8: IMPUTATION QUALITY (SG10K-specific)
# AR2 and DR2 are imputation quality scores — check their distribution
# ════════════════════════════════════════════════════════════════════
print("\n🔬 CHECK 8: IMPUTATION QUALITY (AR2 / DR2)")
print("-" * 60)

ar2_result = subprocess.run(
    f"bcftools query -f '%INFO/AR2\n' {OUTPUT_VCF}",
    shell=True, capture_output=True, text=True
)
dr2_result = subprocess.run(
    f"bcftools query -f '%INFO/DR2\n' {OUTPUT_VCF}",
    shell=True, capture_output=True, text=True
)

ar2_vals = [float(v) for v in ar2_result.stdout.strip().split('\n') if v and v != '.']
dr2_vals = [float(v) for v in dr2_result.stdout.strip().split('\n') if v and v != '.']

if ar2_vals:
    ar2_mean = sum(ar2_vals) / len(ar2_vals)
    ar2_low  = sum(1 for v in ar2_vals if v < 0.3)
    print(f"   AR2 — mean: {ar2_mean:.3f}  low quality (<0.3): {ar2_low:,} ({ar2_low/len(ar2_vals)*100:.1f}%)")
    check("AR2 mean > 0.7 (good imputation quality)", ar2_mean > 0.7, f"Mean AR2 = {ar2_mean:.3f}")
    if ar2_low > 0:
        warn(f"{ar2_low:,} variants have AR2 < 0.3 (low imputation quality)",
             "Consider filtering these out in a later step if needed")
else:
    print("   AR2 field not populated")

if dr2_vals:
    dr2_mean = sum(dr2_vals) / len(dr2_vals)
    dr2_low  = sum(1 for v in dr2_vals if v < 0.3)
    print(f"   DR2 — mean: {dr2_mean:.3f}  low quality (<0.3): {dr2_low:,} ({dr2_low/len(dr2_vals)*100:.1f}%)")
    check("DR2 mean > 0.7 (good imputation quality)", dr2_mean > 0.7, f"Mean DR2 = {dr2_mean:.3f}")
else:
    print("   DR2 field not populated")

# ════════════════════════════════════════════════════════════════════
# CHECK 9: ALLELE FREQUENCY DISTRIBUTION (SG10K-specific)
# AF recalculated for Indian subset — check it makes sense
# ════════════════════════════════════════════════════════════════════
print("\n📈 CHECK 9: ALLELE FREQUENCY DISTRIBUTION")
print("-" * 60)

af_result = subprocess.run(
    f"bcftools query -f '%INFO/AF\n' {OUTPUT_VCF}",
    shell=True, capture_output=True, text=True
)
af_vals = [float(v) for v in af_result.stdout.strip().split('\n') if v and v != '.']

if af_vals:
    af_zero     = sum(1 for v in af_vals if v == 0)
    af_rare     = sum(1 for v in af_vals if 0 < v < 0.01)
    af_common   = sum(1 for v in af_vals if v >= 0.01)
    af_max      = max(af_vals)
    af_mean     = sum(af_vals) / len(af_vals)

    print(f"   AF = 0 (not observed in Indian subset) : {af_zero:,}  ({af_zero/len(af_vals)*100:.1f}%)")
    print(f"   AF > 0 but < 0.01 (rare)               : {af_rare:,}  ({af_rare/len(af_vals)*100:.1f}%)")
    print(f"   AF >= 0.01 (common)                     : {af_common:,} ({af_common/len(af_vals)*100:.1f}%)")
    print(f"   Max AF                                  : {af_max:.4f}")
    print(f"   Mean AF                                 : {af_mean:.6f}")

    check("AF values are between 0 and 1", all(0 <= v <= 1 for v in af_vals))
    check("No AF values above 1.0 (invalid)", all(v <= 1.0 for v in af_vals))
    if af_zero > n_variants * 0.5:
        warn(f"{af_zero:,} variants have AF=0 in Indian subset",
             "These variants exist in gnomAD but were not observed in SG10K Indian samples")

# ════════════════════════════════════════════════════════════════════
# CHECK 10: GENOTYPE FORMAT (SG10K-specific)
# SG10K has actual genotypes — verify FORMAT field and GT is present
# ════════════════════════════════════════════════════════════════════
print("\n🧪 CHECK 10: GENOTYPE FORMAT (SG10K-specific)")
print("-" * 60)

# Check FORMAT field in header
fmt_result = subprocess.run(
    f"bcftools view -h {OUTPUT_VCF} | grep '##FORMAT=<ID=GT'",
    shell=True, capture_output=True, text=True
)
gt_in_header = bool(fmt_result.stdout.strip())
check("GT FORMAT field in header", gt_in_header)

# Check actual genotype columns present
cols_result = subprocess.run(
    f"bcftools view -H {OUTPUT_VCF} | head -1 | awk '{{print NF}}'",
    shell=True, capture_output=True, text=True
)
n_cols = int(cols_result.stdout.strip()) if cols_result.stdout.strip() else 0
expected_cols = 9 + 1125  # 9 fixed VCF columns + 1125 sample columns
print(f"   Columns per row : {n_cols:,} (expected {expected_cols:,})")
check(f"Column count = 9 + 1,125 samples = {expected_cols}", n_cols == expected_cols,
      f"Got {n_cols}, expected {expected_cols}")

# Check a sample genotype looks valid
gt_result = subprocess.run(
    f"bcftools query -f '[%GT ]\\n' {OUTPUT_VCF} | head -1 | tr ' ' '\\n' | sort | uniq -c | sort -rn | head -5",
    shell=True, capture_output=True, text=True
)
print(f"   Most common genotypes (first variant):")
for line in gt_result.stdout.strip().split('\n')[:5]:
    print(f"      {line.strip()}")

# ════════════════════════════════════════════════════════════════════
# CHECK 11: GENE COVERAGE
# ════════════════════════════════════════════════════════════════════
print("\n🧫 CHECK 11: GENE COVERAGE")
print("-" * 60)

gene_counts = defaultdict(int)
protein_pos_missing = 0
amino_acids_missing = 0

proc = subprocess.Popen(f"bcftools view -H {OUTPUT_VCF}", shell=True, stdout=subprocess.PIPE, text=True)
for line in proc.stdout:
    info = line.strip().split('\t')[7]
    if 'CSQ=' not in info:
        continue
    for t in info.split('CSQ=')[1].split(';')[0].split(','):
        cols = t.split('|')
        if len(cols) > CANONICAL_IDX and cols[CANONICAL_IDX] == 'YES':
            if 'missense_variant' in cols[CONSEQUENCE_IDX]:
                gene = cols[SYMBOL_IDX] if len(cols) > SYMBOL_IDX else ''
                gene_counts[gene] += 1
                if not (cols[PROTEIN_POS_IDX] if len(cols) > PROTEIN_POS_IDX else ''):
                    protein_pos_missing += 1
                if not (cols[AMINO_ACIDS_IDX] if len(cols) > AMINO_ACIDS_IDX else ''):
                    amino_acids_missing += 1
                break
proc.wait()

n_genes      = len(gene_counts)
avg_per_gene = n_variants / n_genes if n_genes > 0 else 0
empty_genes  = sum(1 for g in gene_counts if g in ('', 'UNKNOWN'))
top_genes    = sorted(gene_counts.items(), key=lambda x: -x[1])[:5]

print(f"   Unique genes covered        : {n_genes:,}")
print(f"   Avg variants per gene       : {avg_per_gene:.1f}")
print(f"   Empty gene symbols          : {empty_genes:,}")
print(f"   Missing Protein_position    : {protein_pos_missing:,}")
print(f"   Missing Amino_acids         : {amino_acids_missing:,}")
print(f"   Top 5 genes by variant count:")
for gene, count in top_genes:
    print(f"      {gene or '(empty)':20s}: {count:,}")

check("Gene count in plausible range (300–700)", 300 <= n_genes <= 700, f"{n_genes} genes")
check("No excessive empty gene symbols", empty_genes < n_variants * 0.05)
check("Protein_position present for all variants", protein_pos_missing == 0,
      f"{protein_pos_missing:,} missing — needed for sequence generation")
check("Amino_acids present for all variants", amino_acids_missing == 0,
      f"{amino_acids_missing:,} missing — needed for sequence generation")

# ════════════════════════════════════════════════════════════════════
# CHECK 12: DUPLICATE POSITIONS
# ════════════════════════════════════════════════════════════════════
print("\n🔁 CHECK 12: DUPLICATE POSITIONS")
print("-" * 60)

dup_result = subprocess.run(
    f"bcftools query -f '%CHROM:%POS:%REF:%ALT\n' {OUTPUT_VCF} | sort | uniq -d | wc -l",
    shell=True, capture_output=True, text=True
)
n_dups = int(dup_result.stdout.strip()) if dup_result.stdout.strip() else 0
print(f"   Duplicate positions: {n_dups:,}")
check("No duplicate variant positions", n_dups == 0)

# ════════════════════════════════════════════════════════════════════
# CHECK 13: VCF FORMAT INTEGRITY
# ════════════════════════════════════════════════════════════════════
print("\n📋 CHECK 13: VCF FORMAT INTEGRITY")
print("-" * 60)

header_lines = subprocess.run(
    f"bcftools view -h {OUTPUT_VCF} | wc -l",
    shell=True, capture_output=True, text=True
).stdout.strip()
n_header = int(header_lines) if header_lines else 0
print(f"   Header lines: {n_header}")
check("Header present and substantial", n_header > 10)

malformed = subprocess.run(
    f"bcftools view -H {OUTPUT_VCF} | awk 'NF<9' | wc -l",
    shell=True, capture_output=True, text=True
).stdout.strip()
n_malformed = int(malformed) if malformed else 0
check("No malformed variant lines (< 9 columns)", n_malformed == 0,
      f"SG10K needs 9 + 1125 sample columns per line")

empty_alleles = subprocess.run(
    f"bcftools query -f '%REF\t%ALT\n' {OUTPUT_VCF} | awk '$1==\".\" || $2==\".\"' | wc -l",
    shell=True, capture_output=True, text=True
).stdout.strip()
n_empty = int(empty_alleles) if empty_alleles else 0
check("All variants have REF and ALT alleles", n_empty == 0)

# ════════════════════════════════════════════════════════════════════
# FINAL REPORT
# ════════════════════════════════════════════════════════════════════
print("\n" + "=" * 80)
print("QUALITY CHECK REPORT — sg10k_chr22_canonical_missense.vcf.gz")
print("=" * 80)
print(f"\n   ✅ Passed   : {len(passed)}")
print(f"   ❌ Failed   : {len(failed)}")
print(f"   ⚠️  Warnings : {len(warnings)}")

if failed:
    print("\n   Failed checks:")
    for f in failed:
        print(f"      ❌ {f}")
if warnings:
    print("\n   Warnings (non-blocking):")
    for w in warnings:
        print(f"      ⚠️  {w}")

if len(failed) == 0:
    print("""
🎉 ALL CHECKS PASSED!
   sg10k_chr22_canonical_missense.vcf.gz is clean and ready.
   Safe to proceed with remaining chromosomes.
""")
elif len(failed) <= 2:
    print("\n⚠️  MOSTLY PASSED — review failed checks above before proceeding.")
else:
    print("\n❌ MULTIPLE CHECKS FAILED — investigate before proceeding.")

print("=" * 80)

⏳ Filtering canonical missense variants...
   Processed 100,000 variants, kept 818 so far...
   Processed 200,000 variants, kept 1,369 so far...
   Processed 300,000 variants, kept 2,424 so far...
   Processed 400,000 variants, kept 3,328 so far...
   Processed 500,000 variants, kept 3,351 so far...
   Processed 600,000 variants, kept 3,688 so far...
   Processed 700,000 variants, kept 3,800 so far...
   Processed 800,000 variants, kept 4,059 so far...
   Processed 900,000 variants, kept 4,318 so far...
   Processed 1,000,000 variants, kept 4,969 so far...
   Processed 1,100,000 variants, kept 5,289 so far...
   Processed 1,200,000 variants, kept 5,728 so far...
   Processed 1,300,000 variants, kept 6,244 so far...
   Processed 1,400,000 variants, kept 6,466 so far...
   Processed 1,500,000 variants, kept 6,742 so far...
   Processed 1,600,000 variants, kept 7,134 so far...
   Processed 1,700,000 variants, kept 7,400 so far...
   Processed 1,800,000 variants, kept 8,250 so far...
   Pr

In [6]:
import subprocess
import os

chr_num = 11
VCF_FILE = f'/content/drive/MyDrive/FYP_DATA/DATA/annotated_data/sg10k/annotated/sg10k_annotated/sg10k_chr{chr_num}_annotated.vcf.gz'
OUTPUT_DIR = '/content/drive/MyDrive/FYP_DATA/DATA/annotated_data/sg10k/annotated/annotated_and_missense_filtered/'
TEMP_VCF   = OUTPUT_DIR + f'sg10k_chr{chr_num}_canonical_missense.vcf'
OUTPUT_VCF = OUTPUT_DIR + f'sg10k_chr{chr_num}_canonical_missense.vcf.gz'

# CSQ column indices — confirmed from inspection
CONSEQUENCE_IDX = 1
CANONICAL_IDX   = 23

kept  = 0
total = 0

print("⏳ Filtering canonical missense variants...")

with open(TEMP_VCF, 'w') as out_f:

    # ── write header unchanged ──────────────────────────────────
    header_result = subprocess.run(
        f"bcftools view -h {VCF_FILE}",
        shell=True, capture_output=True, text=True
    )
    out_f.write(header_result.stdout)

    # ── stream variants ─────────────────────────────────────────
    proc = subprocess.Popen(
        f"bcftools view -H {VCF_FILE}",
        shell=True, stdout=subprocess.PIPE, text=True
    )

    for line in proc.stdout:
        total += 1
        fields = line.strip().split('\t')
        info   = fields[7]

        if 'CSQ=' not in info:
            continue

        csq_blob    = info.split('CSQ=')[1].split(';')[0]
        transcripts = csq_blob.split(',')

        for transcript in transcripts:
            cols = transcript.split('|')
            if len(cols) <= CANONICAL_IDX:
                continue
            consequence = cols[CONSEQUENCE_IDX]
            canonical   = cols[CANONICAL_IDX]
            if canonical == 'YES' and 'missense_variant' in consequence:
                out_f.write(line)
                kept += 1
                break

        if total % 100_000 == 0:
            print(f"   Processed {total:,} variants, kept {kept:,} so far...")

    proc.wait()

print(f"\n✅ Done! Processed {total:,} total variants")
print(f"   Kept {kept:,} canonical missense variants")

# ── Compress ────────────────────────────────────────────────────
print(f"\n⏳ Compressing...")
bgzip_result = subprocess.run(
    f"bgzip -f {TEMP_VCF}",
    shell=True, capture_output=True, text=True
)
print(f"bgzip return code : {bgzip_result.returncode}")
print(f"bgzip stderr      : {bgzip_result.stderr.strip() or 'none'}")

# ── Index ────────────────────────────────────────────────────────
print(f"\n⏳ Indexing...")
subprocess.run(f"bcftools index -t -f {OUTPUT_VCF}", shell=True, capture_output=True)
print(f"✅ Indexed!")

# ── Verify ───────────────────────────────────────────────────────
print(f"\nCompressed file exists : {os.path.exists(OUTPUT_VCF)}")
print(f"Index file exists      : {os.path.exists(OUTPUT_VCF + '.tbi')}")

count_result = subprocess.run(
    f"bcftools index -n {OUTPUT_VCF}",
    shell=True, capture_output=True, text=True
)
n_variants = int(count_result.stdout.strip())
input_mb   = os.path.getsize(VCF_FILE)  / (1024**2)
output_mb  = os.path.getsize(OUTPUT_VCF) / (1024**2)

print("=" * 60)
print("VERIFICATION")
print("=" * 60)
print(f"Input size             : {input_mb:,.2f} MB")
print(f"Output size            : {output_mb:,.2f} MB")
print(f"Canonical missense kept: {n_variants:,}")
print("=" * 60)

# --------------------------- QUALITY CHECKS ------------------------------ #
import subprocess
import os
from collections import defaultdict

# ── Configuration ────────────────────────────────────────────────────
OUTPUT_VCF = f'/content/drive/MyDrive/FYP_DATA/DATA/annotated_data/sg10k/annotated/annotated_and_missense_filtered/sg10k_chr{chr_num}_canonical_missense.vcf.gz'
VCF_FILE   = f'/content/drive/MyDrive/FYP_DATA/DATA/annotated_data/sg10k/annotated/sg10k_annotated/sg10k_chr{chr_num}_annotated.vcf.gz'

CONSEQUENCE_IDX    = 1
CANONICAL_IDX      = 23
IMPACT_IDX         = 2
SYMBOL_IDX         = 3
PROTEIN_POS_IDX    = 14
AMINO_ACIDS_IDX    = 15

passed = []
failed = []
warnings = []

def check(name, condition, detail=""):
    if condition:
        passed.append(name)
        print(f"   ✅ {name}")
    else:
        failed.append(name)
        print(f"   ❌ {name}")
    if detail:
        print(f"      → {detail}")

def warn(name, detail=""):
    warnings.append(name)
    print(f"   ⚠️  {name}")
    if detail:
        print(f"      → {detail}")

print("=" * 80)
print("QUALITY CHECK — sg10k_chr22_canonical_missense.vcf.gz")
print("=" * 80)

# ════════════════════════════════════════════════════════════════════
# CHECK 1: FILE & INDEX
# ════════════════════════════════════════════════════════════════════
print("\n📁 CHECK 1: FILE & INDEX")
print("-" * 60)

file_ok  = os.path.exists(OUTPUT_VCF)
index_ok = os.path.exists(OUTPUT_VCF + '.tbi')
size_mb  = os.path.getsize(OUTPUT_VCF) / (1024**2) if file_ok else 0

check("File exists",       file_ok,  f"Size: {size_mb:.2f} MB")
check("Index exists",      index_ok)
check("File is not empty", size_mb > 0.1, f"{size_mb:.2f} MB")

# ════════════════════════════════════════════════════════════════════
# CHECK 2: VARIANT COUNT
# ════════════════════════════════════════════════════════════════════
print("\n🔢 CHECK 2: VARIANT COUNT")
print("-" * 60)

count_result = subprocess.run(
    f"bcftools index -n {OUTPUT_VCF}",
    shell=True, capture_output=True, text=True
)
n_variants = int(count_result.stdout.strip()) if count_result.stdout.strip() else 0
print(f"   Total canonical missense variants : {n_variants:,}")

check("Variant count matches filtering output", n_variants == 12_998,
      f"Got {n_variants:,}, expected 12,998")
check("Count is lower than any-transcript missense (14,404)", n_variants < 14_404,
      f"{n_variants:,} < 14,404 ✓ — canonical filter correctly reduced count")
check("Count in plausible range (8k–16k)", 8_000 <= n_variants <= 16_000,
      f"{n_variants:,} variants")

# ════════════════════════════════════════════════════════════════════
# CHECK 3: SAMPLE COUNT
# SG10K-specific — genotypes must be preserved for all 1,125 samples
# ════════════════════════════════════════════════════════════════════
print("\n👥 CHECK 3: SAMPLE COUNT (SG10K-specific)")
print("-" * 60)

samples_result = subprocess.run(
    f"bcftools query -l {OUTPUT_VCF} | wc -l",
    shell=True, capture_output=True, text=True
)
n_samples = int(samples_result.stdout.strip()) if samples_result.stdout.strip() else 0
print(f"   Samples in filtered file : {n_samples:,}")

check("Sample count = 1,125", n_samples == 1_125,
      f"Got {n_samples} — filtering must not remove samples")

# ════════════════════════════════════════════════════════════════════
# CHECK 4: CHROMOSOME INTEGRITY
# ════════════════════════════════════════════════════════════════════
print("\n🧬 CHECK 4: CHROMOSOME INTEGRITY")
print("-" * 60)

chr_result = subprocess.run(
    f"bcftools query -f '%CHROM\n' {OUTPUT_VCF} | sort -u",
    shell=True, capture_output=True, text=True
)
chromosomes = [c for c in chr_result.stdout.strip().split('\n') if c]
print(f"   Chromosomes found: {chromosomes}")

check("Only chr22 present",          chromosomes == ['chr22'], f"Found: {chromosomes}")
check("Chromosome has 'chr' prefix", all(c.startswith('chr') for c in chromosomes))

# ════════════════════════════════════════════════════════════════════
# CHECK 5: CANONICAL MISSENSE INTEGRITY
# Scan every variant — both conditions must be on the SAME transcript
# ════════════════════════════════════════════════════════════════════
print("\n🎯 CHECK 5: CANONICAL MISSENSE INTEGRITY (scanning all variants)")
print("-" * 60)
print("   Scanning every variant...")

non_missense_count  = 0
non_canonical_count = 0
cross_transcript_fp = 0
valid_count         = 0
no_csq_count        = 0

proc = subprocess.Popen(
    f"bcftools view -H {OUTPUT_VCF}",
    shell=True, stdout=subprocess.PIPE, text=True
)
for line in proc.stdout:
    fields = line.strip().split('\t')
    info   = fields[7]

    if 'CSQ=' not in info:
        no_csq_count += 1
        continue

    csq_blob    = info.split('CSQ=')[1].split(';')[0]
    transcripts = csq_blob.split(',')

    found_canonical       = False
    canonical_is_missense = False

    for t in transcripts:
        cols = t.split('|')
        if len(cols) <= CANONICAL_IDX:
            continue
        if cols[CANONICAL_IDX] == 'YES':
            found_canonical = True
            if 'missense_variant' in cols[CONSEQUENCE_IDX]:
                canonical_is_missense = True
                break

    if not found_canonical:
        non_canonical_count += 1
    elif not canonical_is_missense:
        non_missense_count += 1
        for t in transcripts:
            cols = t.split('|')
            if len(cols) > CANONICAL_IDX and cols[CANONICAL_IDX] != 'YES':
                if 'missense_variant' in cols[CONSEQUENCE_IDX]:
                    cross_transcript_fp += 1
                    break
    else:
        valid_count += 1
proc.wait()

print(f"   Fully valid (canonical=YES + missense): {valid_count:,}")
print(f"   No canonical transcript found          : {non_canonical_count:,}")
print(f"   Canonical found but NOT missense       : {non_missense_count:,}")
print(f"   Cross-transcript false positives       : {cross_transcript_fp:,}")
print(f"   No CSQ field                           : {no_csq_count:,}")

check("All variants have CANONICAL=YES transcript",  non_canonical_count == 0)
check("All variants are missense in canonical",       non_missense_count == 0)
check("Zero cross-transcript false positives",        cross_transcript_fp == 0)
check("No variants missing CSQ field",                no_csq_count == 0)
check("Valid count matches total",                    valid_count == n_variants,
      f"{valid_count:,} valid out of {n_variants:,}")

# ════════════════════════════════════════════════════════════════════
# CHECK 6: IMPACT DISTRIBUTION
# All canonical missense should be MODERATE
# ════════════════════════════════════════════════════════════════════
print("\n⚡ CHECK 6: IMPACT DISTRIBUTION")
print("-" * 60)

impact_counts = defaultdict(int)
proc = subprocess.Popen(f"bcftools view -H {OUTPUT_VCF}", shell=True, stdout=subprocess.PIPE, text=True)
for line in proc.stdout:
    info = line.strip().split('\t')[7]
    if 'CSQ=' not in info:
        continue
    for t in info.split('CSQ=')[1].split(';')[0].split(','):
        cols = t.split('|')
        if len(cols) > CANONICAL_IDX and cols[CANONICAL_IDX] == 'YES':
            if 'missense_variant' in cols[CONSEQUENCE_IDX]:
                impact_counts[cols[IMPACT_IDX]] += 1
                break
proc.wait()

print("   Impact breakdown:")
for impact, count in sorted(impact_counts.items(), key=lambda x: -x[1]):
    print(f"      {impact:12s}: {count:,}")

check("All canonical missense are MODERATE impact",
      set(impact_counts.keys()) == {'MODERATE'},
      f"Found: {set(impact_counts.keys())}")

# ════════════════════════════════════════════════════════════════════
# CHECK 7: INFO FIELDS PRESERVED (SG10K-specific)
# AF, AC, AN, AR2, DR2 must all be present
# ════════════════════════════════════════════════════════════════════
print("\n📊 CHECK 7: SG10K INFO FIELDS PRESERVED")
print("-" * 60)

peek = subprocess.run(
    f"bcftools view -H {OUTPUT_VCF} | head -1",
    shell=True, capture_output=True, text=True
).stdout.strip()

if peek:
    info_field = peek.split('\t')[7] if len(peek.split('\t')) > 7 else ''
    for field in ['AF=', 'AC=', 'AN=', 'AR2=', 'DR2=', 'CSQ=']:
        present = field in info_field
        check(f"{field.rstrip('=')} field present", present)
else:
    print("   ❌ Could not read first variant")

# ════════════════════════════════════════════════════════════════════
# CHECK 8: IMPUTATION QUALITY (SG10K-specific)
# AR2 and DR2 are imputation quality scores — check their distribution
# ════════════════════════════════════════════════════════════════════
print("\n🔬 CHECK 8: IMPUTATION QUALITY (AR2 / DR2)")
print("-" * 60)

ar2_result = subprocess.run(
    f"bcftools query -f '%INFO/AR2\n' {OUTPUT_VCF}",
    shell=True, capture_output=True, text=True
)
dr2_result = subprocess.run(
    f"bcftools query -f '%INFO/DR2\n' {OUTPUT_VCF}",
    shell=True, capture_output=True, text=True
)

ar2_vals = [float(v) for v in ar2_result.stdout.strip().split('\n') if v and v != '.']
dr2_vals = [float(v) for v in dr2_result.stdout.strip().split('\n') if v and v != '.']

if ar2_vals:
    ar2_mean = sum(ar2_vals) / len(ar2_vals)
    ar2_low  = sum(1 for v in ar2_vals if v < 0.3)
    print(f"   AR2 — mean: {ar2_mean:.3f}  low quality (<0.3): {ar2_low:,} ({ar2_low/len(ar2_vals)*100:.1f}%)")
    check("AR2 mean > 0.7 (good imputation quality)", ar2_mean > 0.7, f"Mean AR2 = {ar2_mean:.3f}")
    if ar2_low > 0:
        warn(f"{ar2_low:,} variants have AR2 < 0.3 (low imputation quality)",
             "Consider filtering these out in a later step if needed")
else:
    print("   AR2 field not populated")

if dr2_vals:
    dr2_mean = sum(dr2_vals) / len(dr2_vals)
    dr2_low  = sum(1 for v in dr2_vals if v < 0.3)
    print(f"   DR2 — mean: {dr2_mean:.3f}  low quality (<0.3): {dr2_low:,} ({dr2_low/len(dr2_vals)*100:.1f}%)")
    check("DR2 mean > 0.7 (good imputation quality)", dr2_mean > 0.7, f"Mean DR2 = {dr2_mean:.3f}")
else:
    print("   DR2 field not populated")

# ════════════════════════════════════════════════════════════════════
# CHECK 9: ALLELE FREQUENCY DISTRIBUTION (SG10K-specific)
# AF recalculated for Indian subset — check it makes sense
# ════════════════════════════════════════════════════════════════════
print("\n📈 CHECK 9: ALLELE FREQUENCY DISTRIBUTION")
print("-" * 60)

af_result = subprocess.run(
    f"bcftools query -f '%INFO/AF\n' {OUTPUT_VCF}",
    shell=True, capture_output=True, text=True
)
af_vals = [float(v) for v in af_result.stdout.strip().split('\n') if v and v != '.']

if af_vals:
    af_zero     = sum(1 for v in af_vals if v == 0)
    af_rare     = sum(1 for v in af_vals if 0 < v < 0.01)
    af_common   = sum(1 for v in af_vals if v >= 0.01)
    af_max      = max(af_vals)
    af_mean     = sum(af_vals) / len(af_vals)

    print(f"   AF = 0 (not observed in Indian subset) : {af_zero:,}  ({af_zero/len(af_vals)*100:.1f}%)")
    print(f"   AF > 0 but < 0.01 (rare)               : {af_rare:,}  ({af_rare/len(af_vals)*100:.1f}%)")
    print(f"   AF >= 0.01 (common)                     : {af_common:,} ({af_common/len(af_vals)*100:.1f}%)")
    print(f"   Max AF                                  : {af_max:.4f}")
    print(f"   Mean AF                                 : {af_mean:.6f}")

    check("AF values are between 0 and 1", all(0 <= v <= 1 for v in af_vals))
    check("No AF values above 1.0 (invalid)", all(v <= 1.0 for v in af_vals))
    if af_zero > n_variants * 0.5:
        warn(f"{af_zero:,} variants have AF=0 in Indian subset",
             "These variants exist in gnomAD but were not observed in SG10K Indian samples")

# ════════════════════════════════════════════════════════════════════
# CHECK 10: GENOTYPE FORMAT (SG10K-specific)
# SG10K has actual genotypes — verify FORMAT field and GT is present
# ════════════════════════════════════════════════════════════════════
print("\n🧪 CHECK 10: GENOTYPE FORMAT (SG10K-specific)")
print("-" * 60)

# Check FORMAT field in header
fmt_result = subprocess.run(
    f"bcftools view -h {OUTPUT_VCF} | grep '##FORMAT=<ID=GT'",
    shell=True, capture_output=True, text=True
)
gt_in_header = bool(fmt_result.stdout.strip())
check("GT FORMAT field in header", gt_in_header)

# Check actual genotype columns present
cols_result = subprocess.run(
    f"bcftools view -H {OUTPUT_VCF} | head -1 | awk '{{print NF}}'",
    shell=True, capture_output=True, text=True
)
n_cols = int(cols_result.stdout.strip()) if cols_result.stdout.strip() else 0
expected_cols = 9 + 1125  # 9 fixed VCF columns + 1125 sample columns
print(f"   Columns per row : {n_cols:,} (expected {expected_cols:,})")
check(f"Column count = 9 + 1,125 samples = {expected_cols}", n_cols == expected_cols,
      f"Got {n_cols}, expected {expected_cols}")

# Check a sample genotype looks valid
gt_result = subprocess.run(
    f"bcftools query -f '[%GT ]\\n' {OUTPUT_VCF} | head -1 | tr ' ' '\\n' | sort | uniq -c | sort -rn | head -5",
    shell=True, capture_output=True, text=True
)
print(f"   Most common genotypes (first variant):")
for line in gt_result.stdout.strip().split('\n')[:5]:
    print(f"      {line.strip()}")

# ════════════════════════════════════════════════════════════════════
# CHECK 11: GENE COVERAGE
# ════════════════════════════════════════════════════════════════════
print("\n🧫 CHECK 11: GENE COVERAGE")
print("-" * 60)

gene_counts = defaultdict(int)
protein_pos_missing = 0
amino_acids_missing = 0

proc = subprocess.Popen(f"bcftools view -H {OUTPUT_VCF}", shell=True, stdout=subprocess.PIPE, text=True)
for line in proc.stdout:
    info = line.strip().split('\t')[7]
    if 'CSQ=' not in info:
        continue
    for t in info.split('CSQ=')[1].split(';')[0].split(','):
        cols = t.split('|')
        if len(cols) > CANONICAL_IDX and cols[CANONICAL_IDX] == 'YES':
            if 'missense_variant' in cols[CONSEQUENCE_IDX]:
                gene = cols[SYMBOL_IDX] if len(cols) > SYMBOL_IDX else ''
                gene_counts[gene] += 1
                if not (cols[PROTEIN_POS_IDX] if len(cols) > PROTEIN_POS_IDX else ''):
                    protein_pos_missing += 1
                if not (cols[AMINO_ACIDS_IDX] if len(cols) > AMINO_ACIDS_IDX else ''):
                    amino_acids_missing += 1
                break
proc.wait()

n_genes      = len(gene_counts)
avg_per_gene = n_variants / n_genes if n_genes > 0 else 0
empty_genes  = sum(1 for g in gene_counts if g in ('', 'UNKNOWN'))
top_genes    = sorted(gene_counts.items(), key=lambda x: -x[1])[:5]

print(f"   Unique genes covered        : {n_genes:,}")
print(f"   Avg variants per gene       : {avg_per_gene:.1f}")
print(f"   Empty gene symbols          : {empty_genes:,}")
print(f"   Missing Protein_position    : {protein_pos_missing:,}")
print(f"   Missing Amino_acids         : {amino_acids_missing:,}")
print(f"   Top 5 genes by variant count:")
for gene, count in top_genes:
    print(f"      {gene or '(empty)':20s}: {count:,}")

check("Gene count in plausible range (300–700)", 300 <= n_genes <= 700, f"{n_genes} genes")
check("No excessive empty gene symbols", empty_genes < n_variants * 0.05)
check("Protein_position present for all variants", protein_pos_missing == 0,
      f"{protein_pos_missing:,} missing — needed for sequence generation")
check("Amino_acids present for all variants", amino_acids_missing == 0,
      f"{amino_acids_missing:,} missing — needed for sequence generation")

# ════════════════════════════════════════════════════════════════════
# CHECK 12: DUPLICATE POSITIONS
# ════════════════════════════════════════════════════════════════════
print("\n🔁 CHECK 12: DUPLICATE POSITIONS")
print("-" * 60)

dup_result = subprocess.run(
    f"bcftools query -f '%CHROM:%POS:%REF:%ALT\n' {OUTPUT_VCF} | sort | uniq -d | wc -l",
    shell=True, capture_output=True, text=True
)
n_dups = int(dup_result.stdout.strip()) if dup_result.stdout.strip() else 0
print(f"   Duplicate positions: {n_dups:,}")
check("No duplicate variant positions", n_dups == 0)

# ════════════════════════════════════════════════════════════════════
# CHECK 13: VCF FORMAT INTEGRITY
# ════════════════════════════════════════════════════════════════════
print("\n📋 CHECK 13: VCF FORMAT INTEGRITY")
print("-" * 60)

header_lines = subprocess.run(
    f"bcftools view -h {OUTPUT_VCF} | wc -l",
    shell=True, capture_output=True, text=True
).stdout.strip()
n_header = int(header_lines) if header_lines else 0
print(f"   Header lines: {n_header}")
check("Header present and substantial", n_header > 10)

malformed = subprocess.run(
    f"bcftools view -H {OUTPUT_VCF} | awk 'NF<9' | wc -l",
    shell=True, capture_output=True, text=True
).stdout.strip()
n_malformed = int(malformed) if malformed else 0
check("No malformed variant lines (< 9 columns)", n_malformed == 0,
      f"SG10K needs 9 + 1125 sample columns per line")

empty_alleles = subprocess.run(
    f"bcftools query -f '%REF\t%ALT\n' {OUTPUT_VCF} | awk '$1==\".\" || $2==\".\"' | wc -l",
    shell=True, capture_output=True, text=True
).stdout.strip()
n_empty = int(empty_alleles) if empty_alleles else 0
check("All variants have REF and ALT alleles", n_empty == 0)

# ════════════════════════════════════════════════════════════════════
# FINAL REPORT
# ════════════════════════════════════════════════════════════════════
print("\n" + "=" * 80)
print("QUALITY CHECK REPORT — sg10k_chr22_canonical_missense.vcf.gz")
print("=" * 80)
print(f"\n   ✅ Passed   : {len(passed)}")
print(f"   ❌ Failed   : {len(failed)}")
print(f"   ⚠️  Warnings : {len(warnings)}")

if failed:
    print("\n   Failed checks:")
    for f in failed:
        print(f"      ❌ {f}")
if warnings:
    print("\n   Warnings (non-blocking):")
    for w in warnings:
        print(f"      ⚠️  {w}")

if len(failed) == 0:
    print("""
🎉 ALL CHECKS PASSED!
   sg10k_chr22_canonical_missense.vcf.gz is clean and ready.
   Safe to proceed with remaining chromosomes.
""")
elif len(failed) <= 2:
    print("\n⚠️  MOSTLY PASSED — review failed checks above before proceeding.")
else:
    print("\n❌ MULTIPLE CHECKS FAILED — investigate before proceeding.")

print("=" * 80)

⏳ Filtering canonical missense variants...
   Processed 100,000 variants, kept 2,894 so far...
   Processed 200,000 variants, kept 3,916 so far...
   Processed 300,000 variants, kept 5,982 so far...
   Processed 400,000 variants, kept 6,854 so far...
   Processed 500,000 variants, kept 7,375 so far...
   Processed 600,000 variants, kept 7,705 so far...
   Processed 700,000 variants, kept 8,827 so far...
   Processed 800,000 variants, kept 9,104 so far...
   Processed 900,000 variants, kept 9,209 so far...
   Processed 1,000,000 variants, kept 9,410 so far...
   Processed 1,100,000 variants, kept 9,528 so far...
   Processed 1,200,000 variants, kept 10,115 so far...
   Processed 1,300,000 variants, kept 10,686 so far...
   Processed 1,400,000 variants, kept 10,686 so far...
   Processed 1,500,000 variants, kept 10,713 so far...
   Processed 1,600,000 variants, kept 10,990 so far...
   Processed 1,700,000 variants, kept 12,286 so far...
   Processed 1,800,000 variants, kept 12,643 so far

In [7]:
import subprocess
import os

chr_num = 14
VCF_FILE = f'/content/drive/MyDrive/FYP_DATA/DATA/annotated_data/sg10k/annotated/sg10k_annotated/sg10k_chr{chr_num}_annotated.vcf.gz'
OUTPUT_DIR = '/content/drive/MyDrive/FYP_DATA/DATA/annotated_data/sg10k/annotated/annotated_and_missense_filtered/'
TEMP_VCF   = OUTPUT_DIR + f'sg10k_chr{chr_num}_canonical_missense.vcf'
OUTPUT_VCF = OUTPUT_DIR + f'sg10k_chr{chr_num}_canonical_missense.vcf.gz'

# CSQ column indices — confirmed from inspection
CONSEQUENCE_IDX = 1
CANONICAL_IDX   = 23

kept  = 0
total = 0

print("⏳ Filtering canonical missense variants...")

with open(TEMP_VCF, 'w') as out_f:

    # ── write header unchanged ──────────────────────────────────
    header_result = subprocess.run(
        f"bcftools view -h {VCF_FILE}",
        shell=True, capture_output=True, text=True
    )
    out_f.write(header_result.stdout)

    # ── stream variants ─────────────────────────────────────────
    proc = subprocess.Popen(
        f"bcftools view -H {VCF_FILE}",
        shell=True, stdout=subprocess.PIPE, text=True
    )

    for line in proc.stdout:
        total += 1
        fields = line.strip().split('\t')
        info   = fields[7]

        if 'CSQ=' not in info:
            continue

        csq_blob    = info.split('CSQ=')[1].split(';')[0]
        transcripts = csq_blob.split(',')

        for transcript in transcripts:
            cols = transcript.split('|')
            if len(cols) <= CANONICAL_IDX:
                continue
            consequence = cols[CONSEQUENCE_IDX]
            canonical   = cols[CANONICAL_IDX]
            if canonical == 'YES' and 'missense_variant' in consequence:
                out_f.write(line)
                kept += 1
                break

        if total % 100_000 == 0:
            print(f"   Processed {total:,} variants, kept {kept:,} so far...")

    proc.wait()

print(f"\n✅ Done! Processed {total:,} total variants")
print(f"   Kept {kept:,} canonical missense variants")

# ── Compress ────────────────────────────────────────────────────
print(f"\n⏳ Compressing...")
bgzip_result = subprocess.run(
    f"bgzip -f {TEMP_VCF}",
    shell=True, capture_output=True, text=True
)
print(f"bgzip return code : {bgzip_result.returncode}")
print(f"bgzip stderr      : {bgzip_result.stderr.strip() or 'none'}")

# ── Index ────────────────────────────────────────────────────────
print(f"\n⏳ Indexing...")
subprocess.run(f"bcftools index -t -f {OUTPUT_VCF}", shell=True, capture_output=True)
print(f"✅ Indexed!")

# ── Verify ───────────────────────────────────────────────────────
print(f"\nCompressed file exists : {os.path.exists(OUTPUT_VCF)}")
print(f"Index file exists      : {os.path.exists(OUTPUT_VCF + '.tbi')}")

count_result = subprocess.run(
    f"bcftools index -n {OUTPUT_VCF}",
    shell=True, capture_output=True, text=True
)
n_variants = int(count_result.stdout.strip())
input_mb   = os.path.getsize(VCF_FILE)  / (1024**2)
output_mb  = os.path.getsize(OUTPUT_VCF) / (1024**2)

print("=" * 60)
print("VERIFICATION")
print("=" * 60)
print(f"Input size             : {input_mb:,.2f} MB")
print(f"Output size            : {output_mb:,.2f} MB")
print(f"Canonical missense kept: {n_variants:,}")
print("=" * 60)

# --------------------------- QUALITY CHECKS ------------------------------ #
import subprocess
import os
from collections import defaultdict

# ── Configuration ────────────────────────────────────────────────────
OUTPUT_VCF = f'/content/drive/MyDrive/FYP_DATA/DATA/annotated_data/sg10k/annotated/annotated_and_missense_filtered/sg10k_chr{chr_num}_canonical_missense.vcf.gz'
VCF_FILE   = f'/content/drive/MyDrive/FYP_DATA/DATA/annotated_data/sg10k/annotated/sg10k_annotated/sg10k_chr{chr_num}_annotated.vcf.gz'

CONSEQUENCE_IDX    = 1
CANONICAL_IDX      = 23
IMPACT_IDX         = 2
SYMBOL_IDX         = 3
PROTEIN_POS_IDX    = 14
AMINO_ACIDS_IDX    = 15

passed = []
failed = []
warnings = []

def check(name, condition, detail=""):
    if condition:
        passed.append(name)
        print(f"   ✅ {name}")
    else:
        failed.append(name)
        print(f"   ❌ {name}")
    if detail:
        print(f"      → {detail}")

def warn(name, detail=""):
    warnings.append(name)
    print(f"   ⚠️  {name}")
    if detail:
        print(f"      → {detail}")

print("=" * 80)
print("QUALITY CHECK — sg10k_chr22_canonical_missense.vcf.gz")
print("=" * 80)

# ════════════════════════════════════════════════════════════════════
# CHECK 1: FILE & INDEX
# ════════════════════════════════════════════════════════════════════
print("\n📁 CHECK 1: FILE & INDEX")
print("-" * 60)

file_ok  = os.path.exists(OUTPUT_VCF)
index_ok = os.path.exists(OUTPUT_VCF + '.tbi')
size_mb  = os.path.getsize(OUTPUT_VCF) / (1024**2) if file_ok else 0

check("File exists",       file_ok,  f"Size: {size_mb:.2f} MB")
check("Index exists",      index_ok)
check("File is not empty", size_mb > 0.1, f"{size_mb:.2f} MB")

# ════════════════════════════════════════════════════════════════════
# CHECK 2: VARIANT COUNT
# ════════════════════════════════════════════════════════════════════
print("\n🔢 CHECK 2: VARIANT COUNT")
print("-" * 60)

count_result = subprocess.run(
    f"bcftools index -n {OUTPUT_VCF}",
    shell=True, capture_output=True, text=True
)
n_variants = int(count_result.stdout.strip()) if count_result.stdout.strip() else 0
print(f"   Total canonical missense variants : {n_variants:,}")

check("Variant count matches filtering output", n_variants == 12_998,
      f"Got {n_variants:,}, expected 12,998")
check("Count is lower than any-transcript missense (14,404)", n_variants < 14_404,
      f"{n_variants:,} < 14,404 ✓ — canonical filter correctly reduced count")
check("Count in plausible range (8k–16k)", 8_000 <= n_variants <= 16_000,
      f"{n_variants:,} variants")

# ════════════════════════════════════════════════════════════════════
# CHECK 3: SAMPLE COUNT
# SG10K-specific — genotypes must be preserved for all 1,125 samples
# ════════════════════════════════════════════════════════════════════
print("\n👥 CHECK 3: SAMPLE COUNT (SG10K-specific)")
print("-" * 60)

samples_result = subprocess.run(
    f"bcftools query -l {OUTPUT_VCF} | wc -l",
    shell=True, capture_output=True, text=True
)
n_samples = int(samples_result.stdout.strip()) if samples_result.stdout.strip() else 0
print(f"   Samples in filtered file : {n_samples:,}")

check("Sample count = 1,125", n_samples == 1_125,
      f"Got {n_samples} — filtering must not remove samples")

# ════════════════════════════════════════════════════════════════════
# CHECK 4: CHROMOSOME INTEGRITY
# ════════════════════════════════════════════════════════════════════
print("\n🧬 CHECK 4: CHROMOSOME INTEGRITY")
print("-" * 60)

chr_result = subprocess.run(
    f"bcftools query -f '%CHROM\n' {OUTPUT_VCF} | sort -u",
    shell=True, capture_output=True, text=True
)
chromosomes = [c for c in chr_result.stdout.strip().split('\n') if c]
print(f"   Chromosomes found: {chromosomes}")

check("Only chr22 present",          chromosomes == ['chr22'], f"Found: {chromosomes}")
check("Chromosome has 'chr' prefix", all(c.startswith('chr') for c in chromosomes))

# ════════════════════════════════════════════════════════════════════
# CHECK 5: CANONICAL MISSENSE INTEGRITY
# Scan every variant — both conditions must be on the SAME transcript
# ════════════════════════════════════════════════════════════════════
print("\n🎯 CHECK 5: CANONICAL MISSENSE INTEGRITY (scanning all variants)")
print("-" * 60)
print("   Scanning every variant...")

non_missense_count  = 0
non_canonical_count = 0
cross_transcript_fp = 0
valid_count         = 0
no_csq_count        = 0

proc = subprocess.Popen(
    f"bcftools view -H {OUTPUT_VCF}",
    shell=True, stdout=subprocess.PIPE, text=True
)
for line in proc.stdout:
    fields = line.strip().split('\t')
    info   = fields[7]

    if 'CSQ=' not in info:
        no_csq_count += 1
        continue

    csq_blob    = info.split('CSQ=')[1].split(';')[0]
    transcripts = csq_blob.split(',')

    found_canonical       = False
    canonical_is_missense = False

    for t in transcripts:
        cols = t.split('|')
        if len(cols) <= CANONICAL_IDX:
            continue
        if cols[CANONICAL_IDX] == 'YES':
            found_canonical = True
            if 'missense_variant' in cols[CONSEQUENCE_IDX]:
                canonical_is_missense = True
                break

    if not found_canonical:
        non_canonical_count += 1
    elif not canonical_is_missense:
        non_missense_count += 1
        for t in transcripts:
            cols = t.split('|')
            if len(cols) > CANONICAL_IDX and cols[CANONICAL_IDX] != 'YES':
                if 'missense_variant' in cols[CONSEQUENCE_IDX]:
                    cross_transcript_fp += 1
                    break
    else:
        valid_count += 1
proc.wait()

print(f"   Fully valid (canonical=YES + missense): {valid_count:,}")
print(f"   No canonical transcript found          : {non_canonical_count:,}")
print(f"   Canonical found but NOT missense       : {non_missense_count:,}")
print(f"   Cross-transcript false positives       : {cross_transcript_fp:,}")
print(f"   No CSQ field                           : {no_csq_count:,}")

check("All variants have CANONICAL=YES transcript",  non_canonical_count == 0)
check("All variants are missense in canonical",       non_missense_count == 0)
check("Zero cross-transcript false positives",        cross_transcript_fp == 0)
check("No variants missing CSQ field",                no_csq_count == 0)
check("Valid count matches total",                    valid_count == n_variants,
      f"{valid_count:,} valid out of {n_variants:,}")

# ════════════════════════════════════════════════════════════════════
# CHECK 6: IMPACT DISTRIBUTION
# All canonical missense should be MODERATE
# ════════════════════════════════════════════════════════════════════
print("\n⚡ CHECK 6: IMPACT DISTRIBUTION")
print("-" * 60)

impact_counts = defaultdict(int)
proc = subprocess.Popen(f"bcftools view -H {OUTPUT_VCF}", shell=True, stdout=subprocess.PIPE, text=True)
for line in proc.stdout:
    info = line.strip().split('\t')[7]
    if 'CSQ=' not in info:
        continue
    for t in info.split('CSQ=')[1].split(';')[0].split(','):
        cols = t.split('|')
        if len(cols) > CANONICAL_IDX and cols[CANONICAL_IDX] == 'YES':
            if 'missense_variant' in cols[CONSEQUENCE_IDX]:
                impact_counts[cols[IMPACT_IDX]] += 1
                break
proc.wait()

print("   Impact breakdown:")
for impact, count in sorted(impact_counts.items(), key=lambda x: -x[1]):
    print(f"      {impact:12s}: {count:,}")

check("All canonical missense are MODERATE impact",
      set(impact_counts.keys()) == {'MODERATE'},
      f"Found: {set(impact_counts.keys())}")

# ════════════════════════════════════════════════════════════════════
# CHECK 7: INFO FIELDS PRESERVED (SG10K-specific)
# AF, AC, AN, AR2, DR2 must all be present
# ════════════════════════════════════════════════════════════════════
print("\n📊 CHECK 7: SG10K INFO FIELDS PRESERVED")
print("-" * 60)

peek = subprocess.run(
    f"bcftools view -H {OUTPUT_VCF} | head -1",
    shell=True, capture_output=True, text=True
).stdout.strip()

if peek:
    info_field = peek.split('\t')[7] if len(peek.split('\t')) > 7 else ''
    for field in ['AF=', 'AC=', 'AN=', 'AR2=', 'DR2=', 'CSQ=']:
        present = field in info_field
        check(f"{field.rstrip('=')} field present", present)
else:
    print("   ❌ Could not read first variant")

# ════════════════════════════════════════════════════════════════════
# CHECK 8: IMPUTATION QUALITY (SG10K-specific)
# AR2 and DR2 are imputation quality scores — check their distribution
# ════════════════════════════════════════════════════════════════════
print("\n🔬 CHECK 8: IMPUTATION QUALITY (AR2 / DR2)")
print("-" * 60)

ar2_result = subprocess.run(
    f"bcftools query -f '%INFO/AR2\n' {OUTPUT_VCF}",
    shell=True, capture_output=True, text=True
)
dr2_result = subprocess.run(
    f"bcftools query -f '%INFO/DR2\n' {OUTPUT_VCF}",
    shell=True, capture_output=True, text=True
)

ar2_vals = [float(v) for v in ar2_result.stdout.strip().split('\n') if v and v != '.']
dr2_vals = [float(v) for v in dr2_result.stdout.strip().split('\n') if v and v != '.']

if ar2_vals:
    ar2_mean = sum(ar2_vals) / len(ar2_vals)
    ar2_low  = sum(1 for v in ar2_vals if v < 0.3)
    print(f"   AR2 — mean: {ar2_mean:.3f}  low quality (<0.3): {ar2_low:,} ({ar2_low/len(ar2_vals)*100:.1f}%)")
    check("AR2 mean > 0.7 (good imputation quality)", ar2_mean > 0.7, f"Mean AR2 = {ar2_mean:.3f}")
    if ar2_low > 0:
        warn(f"{ar2_low:,} variants have AR2 < 0.3 (low imputation quality)",
             "Consider filtering these out in a later step if needed")
else:
    print("   AR2 field not populated")

if dr2_vals:
    dr2_mean = sum(dr2_vals) / len(dr2_vals)
    dr2_low  = sum(1 for v in dr2_vals if v < 0.3)
    print(f"   DR2 — mean: {dr2_mean:.3f}  low quality (<0.3): {dr2_low:,} ({dr2_low/len(dr2_vals)*100:.1f}%)")
    check("DR2 mean > 0.7 (good imputation quality)", dr2_mean > 0.7, f"Mean DR2 = {dr2_mean:.3f}")
else:
    print("   DR2 field not populated")

# ════════════════════════════════════════════════════════════════════
# CHECK 9: ALLELE FREQUENCY DISTRIBUTION (SG10K-specific)
# AF recalculated for Indian subset — check it makes sense
# ════════════════════════════════════════════════════════════════════
print("\n📈 CHECK 9: ALLELE FREQUENCY DISTRIBUTION")
print("-" * 60)

af_result = subprocess.run(
    f"bcftools query -f '%INFO/AF\n' {OUTPUT_VCF}",
    shell=True, capture_output=True, text=True
)
af_vals = [float(v) for v in af_result.stdout.strip().split('\n') if v and v != '.']

if af_vals:
    af_zero     = sum(1 for v in af_vals if v == 0)
    af_rare     = sum(1 for v in af_vals if 0 < v < 0.01)
    af_common   = sum(1 for v in af_vals if v >= 0.01)
    af_max      = max(af_vals)
    af_mean     = sum(af_vals) / len(af_vals)

    print(f"   AF = 0 (not observed in Indian subset) : {af_zero:,}  ({af_zero/len(af_vals)*100:.1f}%)")
    print(f"   AF > 0 but < 0.01 (rare)               : {af_rare:,}  ({af_rare/len(af_vals)*100:.1f}%)")
    print(f"   AF >= 0.01 (common)                     : {af_common:,} ({af_common/len(af_vals)*100:.1f}%)")
    print(f"   Max AF                                  : {af_max:.4f}")
    print(f"   Mean AF                                 : {af_mean:.6f}")

    check("AF values are between 0 and 1", all(0 <= v <= 1 for v in af_vals))
    check("No AF values above 1.0 (invalid)", all(v <= 1.0 for v in af_vals))
    if af_zero > n_variants * 0.5:
        warn(f"{af_zero:,} variants have AF=0 in Indian subset",
             "These variants exist in gnomAD but were not observed in SG10K Indian samples")

# ════════════════════════════════════════════════════════════════════
# CHECK 10: GENOTYPE FORMAT (SG10K-specific)
# SG10K has actual genotypes — verify FORMAT field and GT is present
# ════════════════════════════════════════════════════════════════════
print("\n🧪 CHECK 10: GENOTYPE FORMAT (SG10K-specific)")
print("-" * 60)

# Check FORMAT field in header
fmt_result = subprocess.run(
    f"bcftools view -h {OUTPUT_VCF} | grep '##FORMAT=<ID=GT'",
    shell=True, capture_output=True, text=True
)
gt_in_header = bool(fmt_result.stdout.strip())
check("GT FORMAT field in header", gt_in_header)

# Check actual genotype columns present
cols_result = subprocess.run(
    f"bcftools view -H {OUTPUT_VCF} | head -1 | awk '{{print NF}}'",
    shell=True, capture_output=True, text=True
)
n_cols = int(cols_result.stdout.strip()) if cols_result.stdout.strip() else 0
expected_cols = 9 + 1125  # 9 fixed VCF columns + 1125 sample columns
print(f"   Columns per row : {n_cols:,} (expected {expected_cols:,})")
check(f"Column count = 9 + 1,125 samples = {expected_cols}", n_cols == expected_cols,
      f"Got {n_cols}, expected {expected_cols}")

# Check a sample genotype looks valid
gt_result = subprocess.run(
    f"bcftools query -f '[%GT ]\\n' {OUTPUT_VCF} | head -1 | tr ' ' '\\n' | sort | uniq -c | sort -rn | head -5",
    shell=True, capture_output=True, text=True
)
print(f"   Most common genotypes (first variant):")
for line in gt_result.stdout.strip().split('\n')[:5]:
    print(f"      {line.strip()}")

# ════════════════════════════════════════════════════════════════════
# CHECK 11: GENE COVERAGE
# ════════════════════════════════════════════════════════════════════
print("\n🧫 CHECK 11: GENE COVERAGE")
print("-" * 60)

gene_counts = defaultdict(int)
protein_pos_missing = 0
amino_acids_missing = 0

proc = subprocess.Popen(f"bcftools view -H {OUTPUT_VCF}", shell=True, stdout=subprocess.PIPE, text=True)
for line in proc.stdout:
    info = line.strip().split('\t')[7]
    if 'CSQ=' not in info:
        continue
    for t in info.split('CSQ=')[1].split(';')[0].split(','):
        cols = t.split('|')
        if len(cols) > CANONICAL_IDX and cols[CANONICAL_IDX] == 'YES':
            if 'missense_variant' in cols[CONSEQUENCE_IDX]:
                gene = cols[SYMBOL_IDX] if len(cols) > SYMBOL_IDX else ''
                gene_counts[gene] += 1
                if not (cols[PROTEIN_POS_IDX] if len(cols) > PROTEIN_POS_IDX else ''):
                    protein_pos_missing += 1
                if not (cols[AMINO_ACIDS_IDX] if len(cols) > AMINO_ACIDS_IDX else ''):
                    amino_acids_missing += 1
                break
proc.wait()

n_genes      = len(gene_counts)
avg_per_gene = n_variants / n_genes if n_genes > 0 else 0
empty_genes  = sum(1 for g in gene_counts if g in ('', 'UNKNOWN'))
top_genes    = sorted(gene_counts.items(), key=lambda x: -x[1])[:5]

print(f"   Unique genes covered        : {n_genes:,}")
print(f"   Avg variants per gene       : {avg_per_gene:.1f}")
print(f"   Empty gene symbols          : {empty_genes:,}")
print(f"   Missing Protein_position    : {protein_pos_missing:,}")
print(f"   Missing Amino_acids         : {amino_acids_missing:,}")
print(f"   Top 5 genes by variant count:")
for gene, count in top_genes:
    print(f"      {gene or '(empty)':20s}: {count:,}")

check("Gene count in plausible range (300–700)", 300 <= n_genes <= 700, f"{n_genes} genes")
check("No excessive empty gene symbols", empty_genes < n_variants * 0.05)
check("Protein_position present for all variants", protein_pos_missing == 0,
      f"{protein_pos_missing:,} missing — needed for sequence generation")
check("Amino_acids present for all variants", amino_acids_missing == 0,
      f"{amino_acids_missing:,} missing — needed for sequence generation")

# ════════════════════════════════════════════════════════════════════
# CHECK 12: DUPLICATE POSITIONS
# ════════════════════════════════════════════════════════════════════
print("\n🔁 CHECK 12: DUPLICATE POSITIONS")
print("-" * 60)

dup_result = subprocess.run(
    f"bcftools query -f '%CHROM:%POS:%REF:%ALT\n' {OUTPUT_VCF} | sort | uniq -d | wc -l",
    shell=True, capture_output=True, text=True
)
n_dups = int(dup_result.stdout.strip()) if dup_result.stdout.strip() else 0
print(f"   Duplicate positions: {n_dups:,}")
check("No duplicate variant positions", n_dups == 0)

# ════════════════════════════════════════════════════════════════════
# CHECK 13: VCF FORMAT INTEGRITY
# ════════════════════════════════════════════════════════════════════
print("\n📋 CHECK 13: VCF FORMAT INTEGRITY")
print("-" * 60)

header_lines = subprocess.run(
    f"bcftools view -h {OUTPUT_VCF} | wc -l",
    shell=True, capture_output=True, text=True
).stdout.strip()
n_header = int(header_lines) if header_lines else 0
print(f"   Header lines: {n_header}")
check("Header present and substantial", n_header > 10)

malformed = subprocess.run(
    f"bcftools view -H {OUTPUT_VCF} | awk 'NF<9' | wc -l",
    shell=True, capture_output=True, text=True
).stdout.strip()
n_malformed = int(malformed) if malformed else 0
check("No malformed variant lines (< 9 columns)", n_malformed == 0,
      f"SG10K needs 9 + 1125 sample columns per line")

empty_alleles = subprocess.run(
    f"bcftools query -f '%REF\t%ALT\n' {OUTPUT_VCF} | awk '$1==\".\" || $2==\".\"' | wc -l",
    shell=True, capture_output=True, text=True
).stdout.strip()
n_empty = int(empty_alleles) if empty_alleles else 0
check("All variants have REF and ALT alleles", n_empty == 0)

# ════════════════════════════════════════════════════════════════════
# FINAL REPORT
# ════════════════════════════════════════════════════════════════════
print("\n" + "=" * 80)
print("QUALITY CHECK REPORT — sg10k_chr22_canonical_missense.vcf.gz")
print("=" * 80)
print(f"\n   ✅ Passed   : {len(passed)}")
print(f"   ❌ Failed   : {len(failed)}")
print(f"   ⚠️  Warnings : {len(warnings)}")

if failed:
    print("\n   Failed checks:")
    for f in failed:
        print(f"      ❌ {f}")
if warnings:
    print("\n   Warnings (non-blocking):")
    for w in warnings:
        print(f"      ⚠️  {w}")

if len(failed) == 0:
    print("""
🎉 ALL CHECKS PASSED!
   sg10k_chr22_canonical_missense.vcf.gz is clean and ready.
   Safe to proceed with remaining chromosomes.
""")
elif len(failed) <= 2:
    print("\n⚠️  MOSTLY PASSED — review failed checks above before proceeding.")
else:
    print("\n❌ MULTIPLE CHECKS FAILED — investigate before proceeding.")

print("=" * 80)

⏳ Filtering canonical missense variants...
   Processed 100,000 variants, kept 1,257 so far...
   Processed 200,000 variants, kept 3,298 so far...
   Processed 300,000 variants, kept 4,462 so far...
   Processed 400,000 variants, kept 4,528 so far...
   Processed 500,000 variants, kept 5,045 so far...
   Processed 600,000 variants, kept 5,421 so far...
   Processed 700,000 variants, kept 5,823 so far...
   Processed 800,000 variants, kept 5,999 so far...
   Processed 900,000 variants, kept 6,038 so far...
   Processed 1,000,000 variants, kept 6,453 so far...
   Processed 1,100,000 variants, kept 6,589 so far...
   Processed 1,200,000 variants, kept 7,361 so far...
   Processed 1,300,000 variants, kept 7,918 so far...
   Processed 1,400,000 variants, kept 8,282 so far...
   Processed 1,500,000 variants, kept 9,002 so far...
   Processed 1,600,000 variants, kept 9,547 so far...
   Processed 1,700,000 variants, kept 10,122 so far...
   Processed 1,800,000 variants, kept 10,860 so far...
 

In [8]:
import subprocess
import os

chr_num = 17
VCF_FILE = f'/content/drive/MyDrive/FYP_DATA/DATA/annotated_data/sg10k/annotated/sg10k_annotated/sg10k_chr{chr_num}_annotated.vcf.gz'
OUTPUT_DIR = '/content/drive/MyDrive/FYP_DATA/DATA/annotated_data/sg10k/annotated/annotated_and_missense_filtered/'
TEMP_VCF   = OUTPUT_DIR + f'sg10k_chr{chr_num}_canonical_missense.vcf'
OUTPUT_VCF = OUTPUT_DIR + f'sg10k_chr{chr_num}_canonical_missense.vcf.gz'

# CSQ column indices — confirmed from inspection
CONSEQUENCE_IDX = 1
CANONICAL_IDX   = 23

kept  = 0
total = 0

print("⏳ Filtering canonical missense variants...")

with open(TEMP_VCF, 'w') as out_f:

    # ── write header unchanged ──────────────────────────────────
    header_result = subprocess.run(
        f"bcftools view -h {VCF_FILE}",
        shell=True, capture_output=True, text=True
    )
    out_f.write(header_result.stdout)

    # ── stream variants ─────────────────────────────────────────
    proc = subprocess.Popen(
        f"bcftools view -H {VCF_FILE}",
        shell=True, stdout=subprocess.PIPE, text=True
    )

    for line in proc.stdout:
        total += 1
        fields = line.strip().split('\t')
        info   = fields[7]

        if 'CSQ=' not in info:
            continue

        csq_blob    = info.split('CSQ=')[1].split(';')[0]
        transcripts = csq_blob.split(',')

        for transcript in transcripts:
            cols = transcript.split('|')
            if len(cols) <= CANONICAL_IDX:
                continue
            consequence = cols[CONSEQUENCE_IDX]
            canonical   = cols[CANONICAL_IDX]
            if canonical == 'YES' and 'missense_variant' in consequence:
                out_f.write(line)
                kept += 1
                break

        if total % 100_000 == 0:
            print(f"   Processed {total:,} variants, kept {kept:,} so far...")

    proc.wait()

print(f"\n✅ Done! Processed {total:,} total variants")
print(f"   Kept {kept:,} canonical missense variants")

# ── Compress ────────────────────────────────────────────────────
print(f"\n⏳ Compressing...")
bgzip_result = subprocess.run(
    f"bgzip -f {TEMP_VCF}",
    shell=True, capture_output=True, text=True
)
print(f"bgzip return code : {bgzip_result.returncode}")
print(f"bgzip stderr      : {bgzip_result.stderr.strip() or 'none'}")

# ── Index ────────────────────────────────────────────────────────
print(f"\n⏳ Indexing...")
subprocess.run(f"bcftools index -t -f {OUTPUT_VCF}", shell=True, capture_output=True)
print(f"✅ Indexed!")

# ── Verify ───────────────────────────────────────────────────────
print(f"\nCompressed file exists : {os.path.exists(OUTPUT_VCF)}")
print(f"Index file exists      : {os.path.exists(OUTPUT_VCF + '.tbi')}")

count_result = subprocess.run(
    f"bcftools index -n {OUTPUT_VCF}",
    shell=True, capture_output=True, text=True
)
n_variants = int(count_result.stdout.strip())
input_mb   = os.path.getsize(VCF_FILE)  / (1024**2)
output_mb  = os.path.getsize(OUTPUT_VCF) / (1024**2)

print("=" * 60)
print("VERIFICATION")
print("=" * 60)
print(f"Input size             : {input_mb:,.2f} MB")
print(f"Output size            : {output_mb:,.2f} MB")
print(f"Canonical missense kept: {n_variants:,}")
print("=" * 60)

# --------------------------- QUALITY CHECKS ------------------------------ #
import subprocess
import os
from collections import defaultdict

# ── Configuration ────────────────────────────────────────────────────
OUTPUT_VCF = f'/content/drive/MyDrive/FYP_DATA/DATA/annotated_data/sg10k/annotated/annotated_and_missense_filtered/sg10k_chr{chr_num}_canonical_missense.vcf.gz'
VCF_FILE   = f'/content/drive/MyDrive/FYP_DATA/DATA/annotated_data/sg10k/annotated/sg10k_annotated/sg10k_chr{chr_num}_annotated.vcf.gz'

CONSEQUENCE_IDX    = 1
CANONICAL_IDX      = 23
IMPACT_IDX         = 2
SYMBOL_IDX         = 3
PROTEIN_POS_IDX    = 14
AMINO_ACIDS_IDX    = 15

passed = []
failed = []
warnings = []

def check(name, condition, detail=""):
    if condition:
        passed.append(name)
        print(f"   ✅ {name}")
    else:
        failed.append(name)
        print(f"   ❌ {name}")
    if detail:
        print(f"      → {detail}")

def warn(name, detail=""):
    warnings.append(name)
    print(f"   ⚠️  {name}")
    if detail:
        print(f"      → {detail}")

print("=" * 80)
print("QUALITY CHECK — sg10k_chr22_canonical_missense.vcf.gz")
print("=" * 80)

# ════════════════════════════════════════════════════════════════════
# CHECK 1: FILE & INDEX
# ════════════════════════════════════════════════════════════════════
print("\n📁 CHECK 1: FILE & INDEX")
print("-" * 60)

file_ok  = os.path.exists(OUTPUT_VCF)
index_ok = os.path.exists(OUTPUT_VCF + '.tbi')
size_mb  = os.path.getsize(OUTPUT_VCF) / (1024**2) if file_ok else 0

check("File exists",       file_ok,  f"Size: {size_mb:.2f} MB")
check("Index exists",      index_ok)
check("File is not empty", size_mb > 0.1, f"{size_mb:.2f} MB")

# ════════════════════════════════════════════════════════════════════
# CHECK 2: VARIANT COUNT
# ════════════════════════════════════════════════════════════════════
print("\n🔢 CHECK 2: VARIANT COUNT")
print("-" * 60)

count_result = subprocess.run(
    f"bcftools index -n {OUTPUT_VCF}",
    shell=True, capture_output=True, text=True
)
n_variants = int(count_result.stdout.strip()) if count_result.stdout.strip() else 0
print(f"   Total canonical missense variants : {n_variants:,}")

check("Variant count matches filtering output", n_variants == 12_998,
      f"Got {n_variants:,}, expected 12,998")
check("Count is lower than any-transcript missense (14,404)", n_variants < 14_404,
      f"{n_variants:,} < 14,404 ✓ — canonical filter correctly reduced count")
check("Count in plausible range (8k–16k)", 8_000 <= n_variants <= 16_000,
      f"{n_variants:,} variants")

# ════════════════════════════════════════════════════════════════════
# CHECK 3: SAMPLE COUNT
# SG10K-specific — genotypes must be preserved for all 1,125 samples
# ════════════════════════════════════════════════════════════════════
print("\n👥 CHECK 3: SAMPLE COUNT (SG10K-specific)")
print("-" * 60)

samples_result = subprocess.run(
    f"bcftools query -l {OUTPUT_VCF} | wc -l",
    shell=True, capture_output=True, text=True
)
n_samples = int(samples_result.stdout.strip()) if samples_result.stdout.strip() else 0
print(f"   Samples in filtered file : {n_samples:,}")

check("Sample count = 1,125", n_samples == 1_125,
      f"Got {n_samples} — filtering must not remove samples")

# ════════════════════════════════════════════════════════════════════
# CHECK 4: CHROMOSOME INTEGRITY
# ════════════════════════════════════════════════════════════════════
print("\n🧬 CHECK 4: CHROMOSOME INTEGRITY")
print("-" * 60)

chr_result = subprocess.run(
    f"bcftools query -f '%CHROM\n' {OUTPUT_VCF} | sort -u",
    shell=True, capture_output=True, text=True
)
chromosomes = [c for c in chr_result.stdout.strip().split('\n') if c]
print(f"   Chromosomes found: {chromosomes}")

check("Only chr22 present",          chromosomes == ['chr22'], f"Found: {chromosomes}")
check("Chromosome has 'chr' prefix", all(c.startswith('chr') for c in chromosomes))

# ════════════════════════════════════════════════════════════════════
# CHECK 5: CANONICAL MISSENSE INTEGRITY
# Scan every variant — both conditions must be on the SAME transcript
# ════════════════════════════════════════════════════════════════════
print("\n🎯 CHECK 5: CANONICAL MISSENSE INTEGRITY (scanning all variants)")
print("-" * 60)
print("   Scanning every variant...")

non_missense_count  = 0
non_canonical_count = 0
cross_transcript_fp = 0
valid_count         = 0
no_csq_count        = 0

proc = subprocess.Popen(
    f"bcftools view -H {OUTPUT_VCF}",
    shell=True, stdout=subprocess.PIPE, text=True
)
for line in proc.stdout:
    fields = line.strip().split('\t')
    info   = fields[7]

    if 'CSQ=' not in info:
        no_csq_count += 1
        continue

    csq_blob    = info.split('CSQ=')[1].split(';')[0]
    transcripts = csq_blob.split(',')

    found_canonical       = False
    canonical_is_missense = False

    for t in transcripts:
        cols = t.split('|')
        if len(cols) <= CANONICAL_IDX:
            continue
        if cols[CANONICAL_IDX] == 'YES':
            found_canonical = True
            if 'missense_variant' in cols[CONSEQUENCE_IDX]:
                canonical_is_missense = True
                break

    if not found_canonical:
        non_canonical_count += 1
    elif not canonical_is_missense:
        non_missense_count += 1
        for t in transcripts:
            cols = t.split('|')
            if len(cols) > CANONICAL_IDX and cols[CANONICAL_IDX] != 'YES':
                if 'missense_variant' in cols[CONSEQUENCE_IDX]:
                    cross_transcript_fp += 1
                    break
    else:
        valid_count += 1
proc.wait()

print(f"   Fully valid (canonical=YES + missense): {valid_count:,}")
print(f"   No canonical transcript found          : {non_canonical_count:,}")
print(f"   Canonical found but NOT missense       : {non_missense_count:,}")
print(f"   Cross-transcript false positives       : {cross_transcript_fp:,}")
print(f"   No CSQ field                           : {no_csq_count:,}")

check("All variants have CANONICAL=YES transcript",  non_canonical_count == 0)
check("All variants are missense in canonical",       non_missense_count == 0)
check("Zero cross-transcript false positives",        cross_transcript_fp == 0)
check("No variants missing CSQ field",                no_csq_count == 0)
check("Valid count matches total",                    valid_count == n_variants,
      f"{valid_count:,} valid out of {n_variants:,}")

# ════════════════════════════════════════════════════════════════════
# CHECK 6: IMPACT DISTRIBUTION
# All canonical missense should be MODERATE
# ════════════════════════════════════════════════════════════════════
print("\n⚡ CHECK 6: IMPACT DISTRIBUTION")
print("-" * 60)

impact_counts = defaultdict(int)
proc = subprocess.Popen(f"bcftools view -H {OUTPUT_VCF}", shell=True, stdout=subprocess.PIPE, text=True)
for line in proc.stdout:
    info = line.strip().split('\t')[7]
    if 'CSQ=' not in info:
        continue
    for t in info.split('CSQ=')[1].split(';')[0].split(','):
        cols = t.split('|')
        if len(cols) > CANONICAL_IDX and cols[CANONICAL_IDX] == 'YES':
            if 'missense_variant' in cols[CONSEQUENCE_IDX]:
                impact_counts[cols[IMPACT_IDX]] += 1
                break
proc.wait()

print("   Impact breakdown:")
for impact, count in sorted(impact_counts.items(), key=lambda x: -x[1]):
    print(f"      {impact:12s}: {count:,}")

check("All canonical missense are MODERATE impact",
      set(impact_counts.keys()) == {'MODERATE'},
      f"Found: {set(impact_counts.keys())}")

# ════════════════════════════════════════════════════════════════════
# CHECK 7: INFO FIELDS PRESERVED (SG10K-specific)
# AF, AC, AN, AR2, DR2 must all be present
# ════════════════════════════════════════════════════════════════════
print("\n📊 CHECK 7: SG10K INFO FIELDS PRESERVED")
print("-" * 60)

peek = subprocess.run(
    f"bcftools view -H {OUTPUT_VCF} | head -1",
    shell=True, capture_output=True, text=True
).stdout.strip()

if peek:
    info_field = peek.split('\t')[7] if len(peek.split('\t')) > 7 else ''
    for field in ['AF=', 'AC=', 'AN=', 'AR2=', 'DR2=', 'CSQ=']:
        present = field in info_field
        check(f"{field.rstrip('=')} field present", present)
else:
    print("   ❌ Could not read first variant")

# ════════════════════════════════════════════════════════════════════
# CHECK 8: IMPUTATION QUALITY (SG10K-specific)
# AR2 and DR2 are imputation quality scores — check their distribution
# ════════════════════════════════════════════════════════════════════
print("\n🔬 CHECK 8: IMPUTATION QUALITY (AR2 / DR2)")
print("-" * 60)

ar2_result = subprocess.run(
    f"bcftools query -f '%INFO/AR2\n' {OUTPUT_VCF}",
    shell=True, capture_output=True, text=True
)
dr2_result = subprocess.run(
    f"bcftools query -f '%INFO/DR2\n' {OUTPUT_VCF}",
    shell=True, capture_output=True, text=True
)

ar2_vals = [float(v) for v in ar2_result.stdout.strip().split('\n') if v and v != '.']
dr2_vals = [float(v) for v in dr2_result.stdout.strip().split('\n') if v and v != '.']

if ar2_vals:
    ar2_mean = sum(ar2_vals) / len(ar2_vals)
    ar2_low  = sum(1 for v in ar2_vals if v < 0.3)
    print(f"   AR2 — mean: {ar2_mean:.3f}  low quality (<0.3): {ar2_low:,} ({ar2_low/len(ar2_vals)*100:.1f}%)")
    check("AR2 mean > 0.7 (good imputation quality)", ar2_mean > 0.7, f"Mean AR2 = {ar2_mean:.3f}")
    if ar2_low > 0:
        warn(f"{ar2_low:,} variants have AR2 < 0.3 (low imputation quality)",
             "Consider filtering these out in a later step if needed")
else:
    print("   AR2 field not populated")

if dr2_vals:
    dr2_mean = sum(dr2_vals) / len(dr2_vals)
    dr2_low  = sum(1 for v in dr2_vals if v < 0.3)
    print(f"   DR2 — mean: {dr2_mean:.3f}  low quality (<0.3): {dr2_low:,} ({dr2_low/len(dr2_vals)*100:.1f}%)")
    check("DR2 mean > 0.7 (good imputation quality)", dr2_mean > 0.7, f"Mean DR2 = {dr2_mean:.3f}")
else:
    print("   DR2 field not populated")

# ════════════════════════════════════════════════════════════════════
# CHECK 9: ALLELE FREQUENCY DISTRIBUTION (SG10K-specific)
# AF recalculated for Indian subset — check it makes sense
# ════════════════════════════════════════════════════════════════════
print("\n📈 CHECK 9: ALLELE FREQUENCY DISTRIBUTION")
print("-" * 60)

af_result = subprocess.run(
    f"bcftools query -f '%INFO/AF\n' {OUTPUT_VCF}",
    shell=True, capture_output=True, text=True
)
af_vals = [float(v) for v in af_result.stdout.strip().split('\n') if v and v != '.']

if af_vals:
    af_zero     = sum(1 for v in af_vals if v == 0)
    af_rare     = sum(1 for v in af_vals if 0 < v < 0.01)
    af_common   = sum(1 for v in af_vals if v >= 0.01)
    af_max      = max(af_vals)
    af_mean     = sum(af_vals) / len(af_vals)

    print(f"   AF = 0 (not observed in Indian subset) : {af_zero:,}  ({af_zero/len(af_vals)*100:.1f}%)")
    print(f"   AF > 0 but < 0.01 (rare)               : {af_rare:,}  ({af_rare/len(af_vals)*100:.1f}%)")
    print(f"   AF >= 0.01 (common)                     : {af_common:,} ({af_common/len(af_vals)*100:.1f}%)")
    print(f"   Max AF                                  : {af_max:.4f}")
    print(f"   Mean AF                                 : {af_mean:.6f}")

    check("AF values are between 0 and 1", all(0 <= v <= 1 for v in af_vals))
    check("No AF values above 1.0 (invalid)", all(v <= 1.0 for v in af_vals))
    if af_zero > n_variants * 0.5:
        warn(f"{af_zero:,} variants have AF=0 in Indian subset",
             "These variants exist in gnomAD but were not observed in SG10K Indian samples")

# ════════════════════════════════════════════════════════════════════
# CHECK 10: GENOTYPE FORMAT (SG10K-specific)
# SG10K has actual genotypes — verify FORMAT field and GT is present
# ════════════════════════════════════════════════════════════════════
print("\n🧪 CHECK 10: GENOTYPE FORMAT (SG10K-specific)")
print("-" * 60)

# Check FORMAT field in header
fmt_result = subprocess.run(
    f"bcftools view -h {OUTPUT_VCF} | grep '##FORMAT=<ID=GT'",
    shell=True, capture_output=True, text=True
)
gt_in_header = bool(fmt_result.stdout.strip())
check("GT FORMAT field in header", gt_in_header)

# Check actual genotype columns present
cols_result = subprocess.run(
    f"bcftools view -H {OUTPUT_VCF} | head -1 | awk '{{print NF}}'",
    shell=True, capture_output=True, text=True
)
n_cols = int(cols_result.stdout.strip()) if cols_result.stdout.strip() else 0
expected_cols = 9 + 1125  # 9 fixed VCF columns + 1125 sample columns
print(f"   Columns per row : {n_cols:,} (expected {expected_cols:,})")
check(f"Column count = 9 + 1,125 samples = {expected_cols}", n_cols == expected_cols,
      f"Got {n_cols}, expected {expected_cols}")

# Check a sample genotype looks valid
gt_result = subprocess.run(
    f"bcftools query -f '[%GT ]\\n' {OUTPUT_VCF} | head -1 | tr ' ' '\\n' | sort | uniq -c | sort -rn | head -5",
    shell=True, capture_output=True, text=True
)
print(f"   Most common genotypes (first variant):")
for line in gt_result.stdout.strip().split('\n')[:5]:
    print(f"      {line.strip()}")

# ════════════════════════════════════════════════════════════════════
# CHECK 11: GENE COVERAGE
# ════════════════════════════════════════════════════════════════════
print("\n🧫 CHECK 11: GENE COVERAGE")
print("-" * 60)

gene_counts = defaultdict(int)
protein_pos_missing = 0
amino_acids_missing = 0

proc = subprocess.Popen(f"bcftools view -H {OUTPUT_VCF}", shell=True, stdout=subprocess.PIPE, text=True)
for line in proc.stdout:
    info = line.strip().split('\t')[7]
    if 'CSQ=' not in info:
        continue
    for t in info.split('CSQ=')[1].split(';')[0].split(','):
        cols = t.split('|')
        if len(cols) > CANONICAL_IDX and cols[CANONICAL_IDX] == 'YES':
            if 'missense_variant' in cols[CONSEQUENCE_IDX]:
                gene = cols[SYMBOL_IDX] if len(cols) > SYMBOL_IDX else ''
                gene_counts[gene] += 1
                if not (cols[PROTEIN_POS_IDX] if len(cols) > PROTEIN_POS_IDX else ''):
                    protein_pos_missing += 1
                if not (cols[AMINO_ACIDS_IDX] if len(cols) > AMINO_ACIDS_IDX else ''):
                    amino_acids_missing += 1
                break
proc.wait()

n_genes      = len(gene_counts)
avg_per_gene = n_variants / n_genes if n_genes > 0 else 0
empty_genes  = sum(1 for g in gene_counts if g in ('', 'UNKNOWN'))
top_genes    = sorted(gene_counts.items(), key=lambda x: -x[1])[:5]

print(f"   Unique genes covered        : {n_genes:,}")
print(f"   Avg variants per gene       : {avg_per_gene:.1f}")
print(f"   Empty gene symbols          : {empty_genes:,}")
print(f"   Missing Protein_position    : {protein_pos_missing:,}")
print(f"   Missing Amino_acids         : {amino_acids_missing:,}")
print(f"   Top 5 genes by variant count:")
for gene, count in top_genes:
    print(f"      {gene or '(empty)':20s}: {count:,}")

check("Gene count in plausible range (300–700)", 300 <= n_genes <= 700, f"{n_genes} genes")
check("No excessive empty gene symbols", empty_genes < n_variants * 0.05)
check("Protein_position present for all variants", protein_pos_missing == 0,
      f"{protein_pos_missing:,} missing — needed for sequence generation")
check("Amino_acids present for all variants", amino_acids_missing == 0,
      f"{amino_acids_missing:,} missing — needed for sequence generation")

# ════════════════════════════════════════════════════════════════════
# CHECK 12: DUPLICATE POSITIONS
# ════════════════════════════════════════════════════════════════════
print("\n🔁 CHECK 12: DUPLICATE POSITIONS")
print("-" * 60)

dup_result = subprocess.run(
    f"bcftools query -f '%CHROM:%POS:%REF:%ALT\n' {OUTPUT_VCF} | sort | uniq -d | wc -l",
    shell=True, capture_output=True, text=True
)
n_dups = int(dup_result.stdout.strip()) if dup_result.stdout.strip() else 0
print(f"   Duplicate positions: {n_dups:,}")
check("No duplicate variant positions", n_dups == 0)

# ════════════════════════════════════════════════════════════════════
# CHECK 13: VCF FORMAT INTEGRITY
# ════════════════════════════════════════════════════════════════════
print("\n📋 CHECK 13: VCF FORMAT INTEGRITY")
print("-" * 60)

header_lines = subprocess.run(
    f"bcftools view -h {OUTPUT_VCF} | wc -l",
    shell=True, capture_output=True, text=True
).stdout.strip()
n_header = int(header_lines) if header_lines else 0
print(f"   Header lines: {n_header}")
check("Header present and substantial", n_header > 10)

malformed = subprocess.run(
    f"bcftools view -H {OUTPUT_VCF} | awk 'NF<9' | wc -l",
    shell=True, capture_output=True, text=True
).stdout.strip()
n_malformed = int(malformed) if malformed else 0
check("No malformed variant lines (< 9 columns)", n_malformed == 0,
      f"SG10K needs 9 + 1125 sample columns per line")

empty_alleles = subprocess.run(
    f"bcftools query -f '%REF\t%ALT\n' {OUTPUT_VCF} | awk '$1==\".\" || $2==\".\"' | wc -l",
    shell=True, capture_output=True, text=True
).stdout.strip()
n_empty = int(empty_alleles) if empty_alleles else 0
check("All variants have REF and ALT alleles", n_empty == 0)

# ════════════════════════════════════════════════════════════════════
# FINAL REPORT
# ════════════════════════════════════════════════════════════════════
print("\n" + "=" * 80)
print("QUALITY CHECK REPORT — sg10k_chr22_canonical_missense.vcf.gz")
print("=" * 80)
print(f"\n   ✅ Passed   : {len(passed)}")
print(f"   ❌ Failed   : {len(failed)}")
print(f"   ⚠️  Warnings : {len(warnings)}")

if failed:
    print("\n   Failed checks:")
    for f in failed:
        print(f"      ❌ {f}")
if warnings:
    print("\n   Warnings (non-blocking):")
    for w in warnings:
        print(f"      ⚠️  {w}")

if len(failed) == 0:
    print("""
🎉 ALL CHECKS PASSED!
   sg10k_chr22_canonical_missense.vcf.gz is clean and ready.
   Safe to proceed with remaining chromosomes.
""")
elif len(failed) <= 2:
    print("\n⚠️  MOSTLY PASSED — review failed checks above before proceeding.")
else:
    print("\n❌ MULTIPLE CHECKS FAILED — investigate before proceeding.")

print("=" * 80)

⏳ Filtering canonical missense variants...
   Processed 100,000 variants, kept 1,361 so far...
   Processed 200,000 variants, kept 2,998 so far...
   Processed 300,000 variants, kept 4,836 so far...
   Processed 400,000 variants, kept 7,328 so far...
   Processed 500,000 variants, kept 8,311 so far...
   Processed 600,000 variants, kept 8,526 so far...
   Processed 700,000 variants, kept 9,553 so far...
   Processed 800,000 variants, kept 10,717 so far...
   Processed 900,000 variants, kept 10,838 so far...
   Processed 1,000,000 variants, kept 12,158 so far...
   Processed 1,100,000 variants, kept 13,132 so far...
   Processed 1,200,000 variants, kept 13,933 so far...
   Processed 1,300,000 variants, kept 15,240 so far...
   Processed 1,400,000 variants, kept 17,436 so far...
   Processed 1,500,000 variants, kept 19,676 so far...
   Processed 1,600,000 variants, kept 20,877 so far...
   Processed 1,700,000 variants, kept 22,286 so far...
   Processed 1,800,000 variants, kept 22,415 so

In [9]:
import subprocess
import os

chr_num = 20
VCF_FILE = f'/content/drive/MyDrive/FYP_DATA/DATA/annotated_data/sg10k/annotated/sg10k_annotated/sg10k_chr{chr_num}_annotated.vcf.gz'
OUTPUT_DIR = '/content/drive/MyDrive/FYP_DATA/DATA/annotated_data/sg10k/annotated/annotated_and_missense_filtered/'
TEMP_VCF   = OUTPUT_DIR + f'sg10k_chr{chr_num}_canonical_missense.vcf'
OUTPUT_VCF = OUTPUT_DIR + f'sg10k_chr{chr_num}_canonical_missense.vcf.gz'

# CSQ column indices — confirmed from inspection
CONSEQUENCE_IDX = 1
CANONICAL_IDX   = 23

kept  = 0
total = 0

print("⏳ Filtering canonical missense variants...")

with open(TEMP_VCF, 'w') as out_f:

    # ── write header unchanged ──────────────────────────────────
    header_result = subprocess.run(
        f"bcftools view -h {VCF_FILE}",
        shell=True, capture_output=True, text=True
    )
    out_f.write(header_result.stdout)

    # ── stream variants ─────────────────────────────────────────
    proc = subprocess.Popen(
        f"bcftools view -H {VCF_FILE}",
        shell=True, stdout=subprocess.PIPE, text=True
    )

    for line in proc.stdout:
        total += 1
        fields = line.strip().split('\t')
        info   = fields[7]

        if 'CSQ=' not in info:
            continue

        csq_blob    = info.split('CSQ=')[1].split(';')[0]
        transcripts = csq_blob.split(',')

        for transcript in transcripts:
            cols = transcript.split('|')
            if len(cols) <= CANONICAL_IDX:
                continue
            consequence = cols[CONSEQUENCE_IDX]
            canonical   = cols[CANONICAL_IDX]
            if canonical == 'YES' and 'missense_variant' in consequence:
                out_f.write(line)
                kept += 1
                break

        if total % 100_000 == 0:
            print(f"   Processed {total:,} variants, kept {kept:,} so far...")

    proc.wait()

print(f"\n✅ Done! Processed {total:,} total variants")
print(f"   Kept {kept:,} canonical missense variants")

# ── Compress ────────────────────────────────────────────────────
print(f"\n⏳ Compressing...")
bgzip_result = subprocess.run(
    f"bgzip -f {TEMP_VCF}",
    shell=True, capture_output=True, text=True
)
print(f"bgzip return code : {bgzip_result.returncode}")
print(f"bgzip stderr      : {bgzip_result.stderr.strip() or 'none'}")

# ── Index ────────────────────────────────────────────────────────
print(f"\n⏳ Indexing...")
subprocess.run(f"bcftools index -t -f {OUTPUT_VCF}", shell=True, capture_output=True)
print(f"✅ Indexed!")

# ── Verify ───────────────────────────────────────────────────────
print(f"\nCompressed file exists : {os.path.exists(OUTPUT_VCF)}")
print(f"Index file exists      : {os.path.exists(OUTPUT_VCF + '.tbi')}")

count_result = subprocess.run(
    f"bcftools index -n {OUTPUT_VCF}",
    shell=True, capture_output=True, text=True
)
n_variants = int(count_result.stdout.strip())
input_mb   = os.path.getsize(VCF_FILE)  / (1024**2)
output_mb  = os.path.getsize(OUTPUT_VCF) / (1024**2)

print("=" * 60)
print("VERIFICATION")
print("=" * 60)
print(f"Input size             : {input_mb:,.2f} MB")
print(f"Output size            : {output_mb:,.2f} MB")
print(f"Canonical missense kept: {n_variants:,}")
print("=" * 60)

# --------------------------- QUALITY CHECKS ------------------------------ #
import subprocess
import os
from collections import defaultdict

# ── Configuration ────────────────────────────────────────────────────
OUTPUT_VCF = f'/content/drive/MyDrive/FYP_DATA/DATA/annotated_data/sg10k/annotated/annotated_and_missense_filtered/sg10k_chr{chr_num}_canonical_missense.vcf.gz'
VCF_FILE   = f'/content/drive/MyDrive/FYP_DATA/DATA/annotated_data/sg10k/annotated/sg10k_annotated/sg10k_chr{chr_num}_annotated.vcf.gz'

CONSEQUENCE_IDX    = 1
CANONICAL_IDX      = 23
IMPACT_IDX         = 2
SYMBOL_IDX         = 3
PROTEIN_POS_IDX    = 14
AMINO_ACIDS_IDX    = 15

passed = []
failed = []
warnings = []

def check(name, condition, detail=""):
    if condition:
        passed.append(name)
        print(f"   ✅ {name}")
    else:
        failed.append(name)
        print(f"   ❌ {name}")
    if detail:
        print(f"      → {detail}")

def warn(name, detail=""):
    warnings.append(name)
    print(f"   ⚠️  {name}")
    if detail:
        print(f"      → {detail}")

print("=" * 80)
print("QUALITY CHECK — sg10k_chr22_canonical_missense.vcf.gz")
print("=" * 80)

# ════════════════════════════════════════════════════════════════════
# CHECK 1: FILE & INDEX
# ════════════════════════════════════════════════════════════════════
print("\n📁 CHECK 1: FILE & INDEX")
print("-" * 60)

file_ok  = os.path.exists(OUTPUT_VCF)
index_ok = os.path.exists(OUTPUT_VCF + '.tbi')
size_mb  = os.path.getsize(OUTPUT_VCF) / (1024**2) if file_ok else 0

check("File exists",       file_ok,  f"Size: {size_mb:.2f} MB")
check("Index exists",      index_ok)
check("File is not empty", size_mb > 0.1, f"{size_mb:.2f} MB")

# ════════════════════════════════════════════════════════════════════
# CHECK 2: VARIANT COUNT
# ════════════════════════════════════════════════════════════════════
print("\n🔢 CHECK 2: VARIANT COUNT")
print("-" * 60)

count_result = subprocess.run(
    f"bcftools index -n {OUTPUT_VCF}",
    shell=True, capture_output=True, text=True
)
n_variants = int(count_result.stdout.strip()) if count_result.stdout.strip() else 0
print(f"   Total canonical missense variants : {n_variants:,}")

check("Variant count matches filtering output", n_variants == 12_998,
      f"Got {n_variants:,}, expected 12,998")
check("Count is lower than any-transcript missense (14,404)", n_variants < 14_404,
      f"{n_variants:,} < 14,404 ✓ — canonical filter correctly reduced count")
check("Count in plausible range (8k–16k)", 8_000 <= n_variants <= 16_000,
      f"{n_variants:,} variants")

# ════════════════════════════════════════════════════════════════════
# CHECK 3: SAMPLE COUNT
# SG10K-specific — genotypes must be preserved for all 1,125 samples
# ════════════════════════════════════════════════════════════════════
print("\n👥 CHECK 3: SAMPLE COUNT (SG10K-specific)")
print("-" * 60)

samples_result = subprocess.run(
    f"bcftools query -l {OUTPUT_VCF} | wc -l",
    shell=True, capture_output=True, text=True
)
n_samples = int(samples_result.stdout.strip()) if samples_result.stdout.strip() else 0
print(f"   Samples in filtered file : {n_samples:,}")

check("Sample count = 1,125", n_samples == 1_125,
      f"Got {n_samples} — filtering must not remove samples")

# ════════════════════════════════════════════════════════════════════
# CHECK 4: CHROMOSOME INTEGRITY
# ════════════════════════════════════════════════════════════════════
print("\n🧬 CHECK 4: CHROMOSOME INTEGRITY")
print("-" * 60)

chr_result = subprocess.run(
    f"bcftools query -f '%CHROM\n' {OUTPUT_VCF} | sort -u",
    shell=True, capture_output=True, text=True
)
chromosomes = [c for c in chr_result.stdout.strip().split('\n') if c]
print(f"   Chromosomes found: {chromosomes}")

check("Only chr22 present",          chromosomes == ['chr22'], f"Found: {chromosomes}")
check("Chromosome has 'chr' prefix", all(c.startswith('chr') for c in chromosomes))

# ════════════════════════════════════════════════════════════════════
# CHECK 5: CANONICAL MISSENSE INTEGRITY
# Scan every variant — both conditions must be on the SAME transcript
# ════════════════════════════════════════════════════════════════════
print("\n🎯 CHECK 5: CANONICAL MISSENSE INTEGRITY (scanning all variants)")
print("-" * 60)
print("   Scanning every variant...")

non_missense_count  = 0
non_canonical_count = 0
cross_transcript_fp = 0
valid_count         = 0
no_csq_count        = 0

proc = subprocess.Popen(
    f"bcftools view -H {OUTPUT_VCF}",
    shell=True, stdout=subprocess.PIPE, text=True
)
for line in proc.stdout:
    fields = line.strip().split('\t')
    info   = fields[7]

    if 'CSQ=' not in info:
        no_csq_count += 1
        continue

    csq_blob    = info.split('CSQ=')[1].split(';')[0]
    transcripts = csq_blob.split(',')

    found_canonical       = False
    canonical_is_missense = False

    for t in transcripts:
        cols = t.split('|')
        if len(cols) <= CANONICAL_IDX:
            continue
        if cols[CANONICAL_IDX] == 'YES':
            found_canonical = True
            if 'missense_variant' in cols[CONSEQUENCE_IDX]:
                canonical_is_missense = True
                break

    if not found_canonical:
        non_canonical_count += 1
    elif not canonical_is_missense:
        non_missense_count += 1
        for t in transcripts:
            cols = t.split('|')
            if len(cols) > CANONICAL_IDX and cols[CANONICAL_IDX] != 'YES':
                if 'missense_variant' in cols[CONSEQUENCE_IDX]:
                    cross_transcript_fp += 1
                    break
    else:
        valid_count += 1
proc.wait()

print(f"   Fully valid (canonical=YES + missense): {valid_count:,}")
print(f"   No canonical transcript found          : {non_canonical_count:,}")
print(f"   Canonical found but NOT missense       : {non_missense_count:,}")
print(f"   Cross-transcript false positives       : {cross_transcript_fp:,}")
print(f"   No CSQ field                           : {no_csq_count:,}")

check("All variants have CANONICAL=YES transcript",  non_canonical_count == 0)
check("All variants are missense in canonical",       non_missense_count == 0)
check("Zero cross-transcript false positives",        cross_transcript_fp == 0)
check("No variants missing CSQ field",                no_csq_count == 0)
check("Valid count matches total",                    valid_count == n_variants,
      f"{valid_count:,} valid out of {n_variants:,}")

# ════════════════════════════════════════════════════════════════════
# CHECK 6: IMPACT DISTRIBUTION
# All canonical missense should be MODERATE
# ════════════════════════════════════════════════════════════════════
print("\n⚡ CHECK 6: IMPACT DISTRIBUTION")
print("-" * 60)

impact_counts = defaultdict(int)
proc = subprocess.Popen(f"bcftools view -H {OUTPUT_VCF}", shell=True, stdout=subprocess.PIPE, text=True)
for line in proc.stdout:
    info = line.strip().split('\t')[7]
    if 'CSQ=' not in info:
        continue
    for t in info.split('CSQ=')[1].split(';')[0].split(','):
        cols = t.split('|')
        if len(cols) > CANONICAL_IDX and cols[CANONICAL_IDX] == 'YES':
            if 'missense_variant' in cols[CONSEQUENCE_IDX]:
                impact_counts[cols[IMPACT_IDX]] += 1
                break
proc.wait()

print("   Impact breakdown:")
for impact, count in sorted(impact_counts.items(), key=lambda x: -x[1]):
    print(f"      {impact:12s}: {count:,}")

check("All canonical missense are MODERATE impact",
      set(impact_counts.keys()) == {'MODERATE'},
      f"Found: {set(impact_counts.keys())}")

# ════════════════════════════════════════════════════════════════════
# CHECK 7: INFO FIELDS PRESERVED (SG10K-specific)
# AF, AC, AN, AR2, DR2 must all be present
# ════════════════════════════════════════════════════════════════════
print("\n📊 CHECK 7: SG10K INFO FIELDS PRESERVED")
print("-" * 60)

peek = subprocess.run(
    f"bcftools view -H {OUTPUT_VCF} | head -1",
    shell=True, capture_output=True, text=True
).stdout.strip()

if peek:
    info_field = peek.split('\t')[7] if len(peek.split('\t')) > 7 else ''
    for field in ['AF=', 'AC=', 'AN=', 'AR2=', 'DR2=', 'CSQ=']:
        present = field in info_field
        check(f"{field.rstrip('=')} field present", present)
else:
    print("   ❌ Could not read first variant")

# ════════════════════════════════════════════════════════════════════
# CHECK 8: IMPUTATION QUALITY (SG10K-specific)
# AR2 and DR2 are imputation quality scores — check their distribution
# ════════════════════════════════════════════════════════════════════
print("\n🔬 CHECK 8: IMPUTATION QUALITY (AR2 / DR2)")
print("-" * 60)

ar2_result = subprocess.run(
    f"bcftools query -f '%INFO/AR2\n' {OUTPUT_VCF}",
    shell=True, capture_output=True, text=True
)
dr2_result = subprocess.run(
    f"bcftools query -f '%INFO/DR2\n' {OUTPUT_VCF}",
    shell=True, capture_output=True, text=True
)

ar2_vals = [float(v) for v in ar2_result.stdout.strip().split('\n') if v and v != '.']
dr2_vals = [float(v) for v in dr2_result.stdout.strip().split('\n') if v and v != '.']

if ar2_vals:
    ar2_mean = sum(ar2_vals) / len(ar2_vals)
    ar2_low  = sum(1 for v in ar2_vals if v < 0.3)
    print(f"   AR2 — mean: {ar2_mean:.3f}  low quality (<0.3): {ar2_low:,} ({ar2_low/len(ar2_vals)*100:.1f}%)")
    check("AR2 mean > 0.7 (good imputation quality)", ar2_mean > 0.7, f"Mean AR2 = {ar2_mean:.3f}")
    if ar2_low > 0:
        warn(f"{ar2_low:,} variants have AR2 < 0.3 (low imputation quality)",
             "Consider filtering these out in a later step if needed")
else:
    print("   AR2 field not populated")

if dr2_vals:
    dr2_mean = sum(dr2_vals) / len(dr2_vals)
    dr2_low  = sum(1 for v in dr2_vals if v < 0.3)
    print(f"   DR2 — mean: {dr2_mean:.3f}  low quality (<0.3): {dr2_low:,} ({dr2_low/len(dr2_vals)*100:.1f}%)")
    check("DR2 mean > 0.7 (good imputation quality)", dr2_mean > 0.7, f"Mean DR2 = {dr2_mean:.3f}")
else:
    print("   DR2 field not populated")

# ════════════════════════════════════════════════════════════════════
# CHECK 9: ALLELE FREQUENCY DISTRIBUTION (SG10K-specific)
# AF recalculated for Indian subset — check it makes sense
# ════════════════════════════════════════════════════════════════════
print("\n📈 CHECK 9: ALLELE FREQUENCY DISTRIBUTION")
print("-" * 60)

af_result = subprocess.run(
    f"bcftools query -f '%INFO/AF\n' {OUTPUT_VCF}",
    shell=True, capture_output=True, text=True
)
af_vals = [float(v) for v in af_result.stdout.strip().split('\n') if v and v != '.']

if af_vals:
    af_zero     = sum(1 for v in af_vals if v == 0)
    af_rare     = sum(1 for v in af_vals if 0 < v < 0.01)
    af_common   = sum(1 for v in af_vals if v >= 0.01)
    af_max      = max(af_vals)
    af_mean     = sum(af_vals) / len(af_vals)

    print(f"   AF = 0 (not observed in Indian subset) : {af_zero:,}  ({af_zero/len(af_vals)*100:.1f}%)")
    print(f"   AF > 0 but < 0.01 (rare)               : {af_rare:,}  ({af_rare/len(af_vals)*100:.1f}%)")
    print(f"   AF >= 0.01 (common)                     : {af_common:,} ({af_common/len(af_vals)*100:.1f}%)")
    print(f"   Max AF                                  : {af_max:.4f}")
    print(f"   Mean AF                                 : {af_mean:.6f}")

    check("AF values are between 0 and 1", all(0 <= v <= 1 for v in af_vals))
    check("No AF values above 1.0 (invalid)", all(v <= 1.0 for v in af_vals))
    if af_zero > n_variants * 0.5:
        warn(f"{af_zero:,} variants have AF=0 in Indian subset",
             "These variants exist in gnomAD but were not observed in SG10K Indian samples")

# ════════════════════════════════════════════════════════════════════
# CHECK 10: GENOTYPE FORMAT (SG10K-specific)
# SG10K has actual genotypes — verify FORMAT field and GT is present
# ════════════════════════════════════════════════════════════════════
print("\n🧪 CHECK 10: GENOTYPE FORMAT (SG10K-specific)")
print("-" * 60)

# Check FORMAT field in header
fmt_result = subprocess.run(
    f"bcftools view -h {OUTPUT_VCF} | grep '##FORMAT=<ID=GT'",
    shell=True, capture_output=True, text=True
)
gt_in_header = bool(fmt_result.stdout.strip())
check("GT FORMAT field in header", gt_in_header)

# Check actual genotype columns present
cols_result = subprocess.run(
    f"bcftools view -H {OUTPUT_VCF} | head -1 | awk '{{print NF}}'",
    shell=True, capture_output=True, text=True
)
n_cols = int(cols_result.stdout.strip()) if cols_result.stdout.strip() else 0
expected_cols = 9 + 1125  # 9 fixed VCF columns + 1125 sample columns
print(f"   Columns per row : {n_cols:,} (expected {expected_cols:,})")
check(f"Column count = 9 + 1,125 samples = {expected_cols}", n_cols == expected_cols,
      f"Got {n_cols}, expected {expected_cols}")

# Check a sample genotype looks valid
gt_result = subprocess.run(
    f"bcftools query -f '[%GT ]\\n' {OUTPUT_VCF} | head -1 | tr ' ' '\\n' | sort | uniq -c | sort -rn | head -5",
    shell=True, capture_output=True, text=True
)
print(f"   Most common genotypes (first variant):")
for line in gt_result.stdout.strip().split('\n')[:5]:
    print(f"      {line.strip()}")

# ════════════════════════════════════════════════════════════════════
# CHECK 11: GENE COVERAGE
# ════════════════════════════════════════════════════════════════════
print("\n🧫 CHECK 11: GENE COVERAGE")
print("-" * 60)

gene_counts = defaultdict(int)
protein_pos_missing = 0
amino_acids_missing = 0

proc = subprocess.Popen(f"bcftools view -H {OUTPUT_VCF}", shell=True, stdout=subprocess.PIPE, text=True)
for line in proc.stdout:
    info = line.strip().split('\t')[7]
    if 'CSQ=' not in info:
        continue
    for t in info.split('CSQ=')[1].split(';')[0].split(','):
        cols = t.split('|')
        if len(cols) > CANONICAL_IDX and cols[CANONICAL_IDX] == 'YES':
            if 'missense_variant' in cols[CONSEQUENCE_IDX]:
                gene = cols[SYMBOL_IDX] if len(cols) > SYMBOL_IDX else ''
                gene_counts[gene] += 1
                if not (cols[PROTEIN_POS_IDX] if len(cols) > PROTEIN_POS_IDX else ''):
                    protein_pos_missing += 1
                if not (cols[AMINO_ACIDS_IDX] if len(cols) > AMINO_ACIDS_IDX else ''):
                    amino_acids_missing += 1
                break
proc.wait()

n_genes      = len(gene_counts)
avg_per_gene = n_variants / n_genes if n_genes > 0 else 0
empty_genes  = sum(1 for g in gene_counts if g in ('', 'UNKNOWN'))
top_genes    = sorted(gene_counts.items(), key=lambda x: -x[1])[:5]

print(f"   Unique genes covered        : {n_genes:,}")
print(f"   Avg variants per gene       : {avg_per_gene:.1f}")
print(f"   Empty gene symbols          : {empty_genes:,}")
print(f"   Missing Protein_position    : {protein_pos_missing:,}")
print(f"   Missing Amino_acids         : {amino_acids_missing:,}")
print(f"   Top 5 genes by variant count:")
for gene, count in top_genes:
    print(f"      {gene or '(empty)':20s}: {count:,}")

check("Gene count in plausible range (300–700)", 300 <= n_genes <= 700, f"{n_genes} genes")
check("No excessive empty gene symbols", empty_genes < n_variants * 0.05)
check("Protein_position present for all variants", protein_pos_missing == 0,
      f"{protein_pos_missing:,} missing — needed for sequence generation")
check("Amino_acids present for all variants", amino_acids_missing == 0,
      f"{amino_acids_missing:,} missing — needed for sequence generation")

# ════════════════════════════════════════════════════════════════════
# CHECK 12: DUPLICATE POSITIONS
# ════════════════════════════════════════════════════════════════════
print("\n🔁 CHECK 12: DUPLICATE POSITIONS")
print("-" * 60)

dup_result = subprocess.run(
    f"bcftools query -f '%CHROM:%POS:%REF:%ALT\n' {OUTPUT_VCF} | sort | uniq -d | wc -l",
    shell=True, capture_output=True, text=True
)
n_dups = int(dup_result.stdout.strip()) if dup_result.stdout.strip() else 0
print(f"   Duplicate positions: {n_dups:,}")
check("No duplicate variant positions", n_dups == 0)

# ════════════════════════════════════════════════════════════════════
# CHECK 13: VCF FORMAT INTEGRITY
# ════════════════════════════════════════════════════════════════════
print("\n📋 CHECK 13: VCF FORMAT INTEGRITY")
print("-" * 60)

header_lines = subprocess.run(
    f"bcftools view -h {OUTPUT_VCF} | wc -l",
    shell=True, capture_output=True, text=True
).stdout.strip()
n_header = int(header_lines) if header_lines else 0
print(f"   Header lines: {n_header}")
check("Header present and substantial", n_header > 10)

malformed = subprocess.run(
    f"bcftools view -H {OUTPUT_VCF} | awk 'NF<9' | wc -l",
    shell=True, capture_output=True, text=True
).stdout.strip()
n_malformed = int(malformed) if malformed else 0
check("No malformed variant lines (< 9 columns)", n_malformed == 0,
      f"SG10K needs 9 + 1125 sample columns per line")

empty_alleles = subprocess.run(
    f"bcftools query -f '%REF\t%ALT\n' {OUTPUT_VCF} | awk '$1==\".\" || $2==\".\"' | wc -l",
    shell=True, capture_output=True, text=True
).stdout.strip()
n_empty = int(empty_alleles) if empty_alleles else 0
check("All variants have REF and ALT alleles", n_empty == 0)

# ════════════════════════════════════════════════════════════════════
# FINAL REPORT
# ════════════════════════════════════════════════════════════════════
print("\n" + "=" * 80)
print("QUALITY CHECK REPORT — sg10k_chr22_canonical_missense.vcf.gz")
print("=" * 80)
print(f"\n   ✅ Passed   : {len(passed)}")
print(f"   ❌ Failed   : {len(failed)}")
print(f"   ⚠️  Warnings : {len(warnings)}")

if failed:
    print("\n   Failed checks:")
    for f in failed:
        print(f"      ❌ {f}")
if warnings:
    print("\n   Warnings (non-blocking):")
    for w in warnings:
        print(f"      ⚠️  {w}")

if len(failed) == 0:
    print("""
🎉 ALL CHECKS PASSED!
   sg10k_chr22_canonical_missense.vcf.gz is clean and ready.
   Safe to proceed with remaining chromosomes.
""")
elif len(failed) <= 2:
    print("\n⚠️  MOSTLY PASSED — review failed checks above before proceeding.")
else:
    print("\n❌ MULTIPLE CHECKS FAILED — investigate before proceeding.")

print("=" * 80)

⏳ Filtering canonical missense variants...
   Processed 100,000 variants, kept 896 so far...
   Processed 200,000 variants, kept 1,989 so far...
   Processed 300,000 variants, kept 2,261 so far...
   Processed 400,000 variants, kept 2,594 so far...
   Processed 500,000 variants, kept 2,793 so far...
   Processed 600,000 variants, kept 2,921 so far...
   Processed 700,000 variants, kept 3,379 so far...
   Processed 800,000 variants, kept 3,778 so far...
   Processed 900,000 variants, kept 4,267 so far...
   Processed 1,000,000 variants, kept 4,396 so far...
   Processed 1,100,000 variants, kept 5,439 so far...
   Processed 1,200,000 variants, kept 6,745 so far...
   Processed 1,300,000 variants, kept 7,285 so far...
   Processed 1,400,000 variants, kept 7,815 so far...
   Processed 1,500,000 variants, kept 9,156 so far...
   Processed 1,600,000 variants, kept 9,790 so far...
   Processed 1,700,000 variants, kept 10,548 so far...
   Processed 1,800,000 variants, kept 10,855 so far...
   